# Introduction

TODO:
* Item Effects (Gift type by Faction or Theme preference?)

* Determining resource spawning: success, location, success, types, quality

* Island Health and Growth Potions

* Resource Spawning Overrides (Growth Potions, Imbalance Events, Rare Resource Migrations)


* rename "name" to "resource_name", "creature_name" etc


OPEN QUESTIONS:
* Requisition Board ()
* Newspaper
* Creatures Activities/Themes/Events
* Collective: Bridges/ Murals/ Monument
* Daily Events (Cookie Party)
* Player Progression / Badges / Individual Accomplishments?
* Island Health
* Bridges / Anomalies

# 1. World Time, Weather and Map

## 1a. **World Time**


**Overview:**

The World Clock manages the passage of time within the game. There are four distinct times of day, a seven-day week cycle, and a cyclical four-season year.

The time of day, day of week, and season will influence weather patterns, resource spawning rates, and creature behaviors and preferences.

**Specifications:**

1. **Times of Day:**  
   - The day is divided into four segments:
     1. "AM Early"
     2. "AM Late"
     3. "PM Early"
     4. "PM Late"

2. **Days and Weekdays:**
   - Each "day" is represented by an incrementing integer (Day 0, Day 1, Day 2, and so on).
   - Days are sequentially ordered into Weekdays from "Monday" to "Sunday". The game will continually cycle through this seven-day order.

3. **Seasons:**
   - The game world has four seasons: Spring, Summer, Fall, and Winter.
   - Each season lasts for a duration of 28 days.
   - As the game progresses, seasons will cycle in the given order.
   - Seasons can affect Weather patterns. These have no associated art changes, and just affect resource spawning rates.

**Effects on Other Game Systems:**

1. **Resource Spawning:**
   - The probability of any given type of resource spawning will depend on the time of day (along with other things). Example: "bug" type resources are more likely to spawn in the Early AM.

2. **Creature Behaviors:**
   - Creatures have different movement and activity patterns depending on time of day and day of the week. Some creatures might be most likely to rest during "PM Late", while others might do an activity during "PM Early".

3. **Weather Patterns:**
   - The island weather patterns can be influenced by season.

**Implementation:**

The `WorldClock` class below illustrates the progression of time in the game. This class tracks the current time of day, the day of the week, and the current season. The class provides functions to retrieve the current day, day of the week, season, and time of day. A method named `next()` progresses the time by one segment.

**Open Questions:**

1. Does art or lighting change based on time of day (or season)?

### WorldClock class

In [89]:
class WorldClock:
    DAYS_IN_SEASON = 28  # Each season has 28 days

    def __init__(self, start_day):
        self._times_of_day = ["AM Early", "AM Late", "PM Early", "PM Late"]
        self._days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        self._seasons = ["Spring", "Summer", "Fall", "Winter"]

        self._current_time = 0
        self._day = start_day
        self._day_of_week_index = start_day % len(self._days_of_week)

    def get_current_time(self):
        """Return the current time index."""
        return self._current_time

    def get_time_of_day(self):
        """Return the current time of day as a string."""
        return self._times_of_day[self._current_time % len(self._times_of_day)]

    def get_day(self):
        """Return the current day index."""
        return self._day

    def get_day_of_week(self):
        """Return the current day of the week as a string."""
        return self._days_of_week[self._day % len(self._days_of_week)]

    def get_season(self):
        """Return the current season as a string."""
        return self._seasons[self._day // WorldClock.DAYS_IN_SEASON % len(self._seasons)]

    def next(self):
        """Move the clock to the next time of day."""
        self._current_time = (self._current_time + 1) % len(self._times_of_day)
        if self.get_time_of_day() == "AM Early":
            self._day += 1

## 1b. **Weather**

**Overview:**

The Weather system introduces an element of unpredictability and strategic planning to the game by simulating fluctuating weather conditions. Using a combination of randomized rain and temperature patterns, players will experience a changing environment that affects gameplay. This dynamic feature enriches the world's realism and adds layers of complexity to player decisions.

**Design Goal:**

The Weather will affect creature patterns, resource spawning patterns, and/or creature item effects. Knowing the effect of weather on these variables should allow the player to adapt decisions (e.g. give different gifts on rainy days or forage in different areas), or to see patterns in the weather that let them predict what the weather will be (i.e. "forecasting"), to prepare (i.e. weather as a variable in "demand prediction").

**Specifications:**

1. **Weather: Rainy/Sunny:**  
   - At any given time, it can be either "Rainy" or "Sunny".
   - These are currently set randomly, so there is no prediction capability.

2. **Temperature: Warm/Cool:**  
   - The game's ambient temperature can be either "Cool" or "Warm".
   - These are currently set randomly, so there is no prediction capability.

**Effects on Gameplay:**

1. **Resource Availability:**
   - The weather directly impacts the availability of resources. Some resources might become scarce during rainy days, while others may be more likely to spawn in warm temperatures.

2. **Creature Behaviors:**  
   - Creature behaviors (movements around the map, or item preferences) can change based on weather. For example, creatures could stay on their home island when it is Rainy, or could prefer ice creams on Warm days. Currently there is no effect.

**Implementation:**

The `Weather` class is responsible for managing the game's weather system. Upon instantiation, it randomly sets the initial rain and temperature conditions. The class provides methods to retrieve the current rain status, and current temperature. Every time the `next()` method is invoked, the system randomly updates the rain and temperature conditions.

**Open Questions:**
1. Are there weather patterns across the islands? Or per island? Applied to a subset of screens (raining on one screen and not others on the island)?
2. Is there a seasonal dependence of weather? Time of day?

### Weather class

In [90]:
import random

class Weather:
    RAIN_CONDITIONS = ["Rainy", "Sunny"]
    TEMPERATURES = ["Cool", "Warm"]

    def __init__(self):
        """Initializes the Weather instance with random rain and temperature conditions."""
        self._rain_condition = random.choice(Weather.RAIN_CONDITIONS)
        self._temp = random.choice(Weather.TEMPERATURES)

    def get_rain_condition(self):
        """Return the current rain condition."""
        return self._rain_condition

    def get_temp(self):
        """Return the current temperature condition."""
        return self._temp

    def next(self):
        """Updates the weather conditions to a new random state."""
        self._rain_condition = random.choice(Weather.RAIN_CONDITIONS)
        self._temp = random.choice(Weather.TEMPERATURES)

## 1c. **Map**

**Overview:**

The game world is defined through a nested hierarchy: World > Islands > Plots > Patches > Nodes. Nodes represent individual tiles where resources can spawn. Each level in the hierarchy has associated level variables that are passed down to the lower levels, and can be used to determine spawn rate, probabilities and/or creature motions.

**Hierarchy Levels:**

1. **World**:
    * The largest encompassing structure.
    * Contains `Clock`, `Weather`, and `Islands`.
    * Method `next()` increments both time and weather.

2. **Island**:
    * There are 4 Islands in the World.
    * Every Island has a unique `name` and `offset` for positioning.
    * Each Island consists of multiple `Plots`.

3. **Plot**:
    * Plots are smaller areas within an Island.
    * Plots contain smaller groups of tiles called `Patches`.
    * Each Plot has a unique `name`, an `offset`, and specific layouts of the patches within them.
    * Plots each have a `shade_level` (static, does not change) and `light_color` attributes. The light_color can be changed. A plot has an `is_anomalous` variable, hidden from the player, that tracks whether an anomaly has been placed on the plot.
    * Plots can be accessible to individual players or not, depending on the player friendship level with a group or subgroup of Creatures of that island.
    * Plots have `pastime_types` associated with different creature behaviors or activities. For example, some plots may be places that creatures can "rest", "hangout", "tend" or do an "activity." When creatures select one of the above pastimes, they then move to a plot that allows that pastime. These variables are defined for each plot, but the logic is defined in the **Creature Behavior** section.

4. **Patch**:
    * Patches are contained within Plots, and consist of individual `Nodes`.
    * Every Patch has a specific layout type, and an `offset` from its parent Plot.
    * The state of a Patch is defined by its `water_level` and `patch_health`. Patch health updates when resources are foraged from that patch by a player. Patch health can update overnight depending on creature stats (described in the section on **Update Logic**.)

5. **Node**:
    * Nodes represent a single game tile.
    * A Node belongs to a Patch.
    * A Node has an `offset` from its parent Patch, and a `coords` variable found by adding up all offset vectors from its parent patch, plot, and island.
    * Nodes spawn resources that can be foraged by the player. The type and quality of resources spawned are probabilistic and can depend on any combination of nested variables passed down to the node. These are defined in the section on **Spawn Behavior**.
    * The Node can have a node "type" that determines its visual appearance, user interaction, and resource spawn behavior.
      * Ex1 Node Type: "Overturned Rock". The player overturns a rock to see what resource may be under. There is a chance that there is no resource at all. If there is a resource found underneath, the resource may be more likely to have spawn behavior of 'bug' or 'rock' type.
      * Ex2 Node Type: "Flower Blossom". The player opens the petals of a flower to forage the resource contained inside. The player always recieves a resource, and only the resource type varies. The resource may be more likely to have spawn behavior of 'plant' or 'mushroom' type.

**Interactivity & Data Flow:**

- **Downward Flow**: Higher hierarchy levels pass variables down to their children. For example, the weather from the World and the Island faction can influence type or quality of resource spawned at the Node level.
  
- **Upward Feedback**: Lower hierarchy levels can pass data or trigger events that affect parent structures. For example, foraging resources at the Node level might decrease the health of its parent Patch, contributing to a decrease in Island Health.

**Effects on Gameplay:**

1. **Strategic Foraging**: Nested variables affect the probabilities of spawning different types of resources at different locations within the game.

2. **Environmental Control**: Players can influence variables at the Plot and Patch levels:
  * setting the `color` on a plot (TBD)
  * watering or applying growth potions to alter a patch's `water_level` or `patch_health`. Selecting a patch and altering these variables allows players to improve their chances of getting desired resources.

3. **Foraging Effects**: The hierarchical structure allows for events or actions at one level to bubble up to affect higher levels. When players forage from a Node, this affects the Patch's `patch_health` and the island's aggregate `island_health`.

**Open Questions:**

1. Does player foraging affect Island Health directly, via aggregating effects on `patch_health`?
2. Does `resource_type` or `resource_faction` of the resource foraged matter (or, its `resource_quality`?), or just the total number of foraging events?

## Nested Components Code

In [91]:
import random
import copy

class NestedComponent:
    def __init__(self):
        self.level = ""
        self.children = []
        self.mods = []  # List of weighting functions to apply
        self.base_weights = []
        self.weights = []

    def get_level_variables(self):
        """
        Returns a dictionary of level variables.
        Derived classes should override this method with their own variables.
        """
        return {}

    def get_random_child(self):
        """
        Returns a random child from the children list.
        """
        return random.choice(self.children)

    def update_from_base_weights(self, base_weights):
        """
        Updates weights from base weights, applies modifications, and propagates the weights.
        """
        self.base_weights = copy.deepcopy(base_weights)
        self.weights = copy.deepcopy(base_weights)

        self.apply_modifications()
        self.propagate_weights_to_children()

    def apply_modifications(self):
        """
        Applies modifications to the weights based on the defined mods.
        """
        level_vars = self.get_level_variables()
        for mod_function in self.mods:
            self.weights = mod_function(self.weights, level_vars)

    def propagate_weights_to_children(self):
        """
        Propagates weights to each child in the children list.
        """
        for child in self.children:
            child.update_from_base_weights(self.weights)

### 1c1. Nodes

A `Node` represents the smallest individual unit of a patch. It can be seen as the elementary building block for our island world. Nodes might contain resources, and have specific coordinates denoting their position.

- **Properties**:
    - `coords`: The exact coordinates of the node in the world.
    - `resource`: Resource information associated with the node.

In [92]:
import random

class Node(NestedComponent):
    def __init__(self, parent_patch, node_data, mods):
        super().__init__()  # Initialize parent class
        self.level = "node"

        self.parent_patch = parent_patch
        plot = parent_patch.parent_plot
        island = plot.get_parent_island()

        self.mods = mods.get(self.level, {})

        # Combine offsets for x and y coordinates
        x_offset = sum([island.offset[0], plot.offset[0], parent_patch.offset[0], node_data["offset"][0]])
        y_offset = sum([island.offset[1], plot.offset[1], parent_patch.offset[1], node_data["offset"][1]])
        self.coords = (x_offset, y_offset)

        self.resource = {}

    def get_level_variables(self):
        return {
            "coords": self.coords
        }

    def get_weights(self):
        return self.weights

    def normalize_probabilities(self, base_prob, adjustments):
        """Adjust and normalize probabilities."""
        adjusted_prob = base_prob.copy()
        for key, value in adjustments.items():
            adjusted_prob[key] *= value

        total = sum(adjusted_prob.values())
        for key in adjusted_prob:
            adjusted_prob[key] /= total

        return adjusted_prob

    def spawn_resource(self):
        """Generate a resource based on weights and assign a quality to it."""
        resources = self.weights
        weights = [resource["weight"] for resource in resources]
        resource = random.choices(resources, weights, k=1)[0]

        resource["resource_quality"] = random.randint(50, 100)
        return resource

### 1c2. Patches

The `Patch` class represents a smaller portion of a plot. A patch might have properties such as `water_level` and `soil_quality`. Each patch further divides into multiple nodes.

- **Properties**:
    - `layout`: The type of layout a patch has (small, medium, large).
    - `children`: List of child nodes in the patch.

In [93]:
import random

# Nodes and offsets, for different Patch layouts
PATCH_NODES = {
    "small_patch": [(0,1), (1,1), (0,0), (1,0)],                                      # 2x2 patch
    "medium_patch": [(0,1), (1,1), (2,1), (0,0), (1,0), (2,0)],                       # 3x2 patch
    "large_patch": [(0,2), (1,2), (2,2), (0,1), (1,1), (2,1), (0,0), (1,0), (2,0)]    # 3x3 patch
}

class Patch(NestedComponent):
    DEFAULT_LAYOUT = "small_patch"

    def __init__(self, parent_plot, patch_data, mods):
        """
        Represents a patch layout with attributes such as water level and soil quality.
        """
        super().__init__()  # Initialize parent class
        self.level = "patch"
        self.parent_plot = parent_plot
        self.mods = mods.get(self.level, {})

        # Initialize patch attributes
        self.layout = patch_data.get("layout", Patch.DEFAULT_LAYOUT)
        self.offset = patch_data.get("offset", (0,0))
        self.water_level = patch_data.get("water_level", random.randint(0, 100))
        self.soil_quality = patch_data.get("soil_quality", random.randint(0, 100))

        # Initialize child nodes for the patch
        if self.layout not in PATCH_NODES:
            raise ValueError(f"Unknown patch layout: {self.layout}")
        self.children = [Node(self, {"offset": node_offset}, mods) for node_offset in PATCH_NODES[self.layout]]

    def get_level_variables(self):
        """Return the patch's water level and soil quality."""
        return {
            "water_level": self.water_level,
            "soil_quality": self.soil_quality
        }

### 1a3. Plots

The `Plot` class describes a section of land on an island, comprising properties like `shade_level`, `light_color`, and others. Each plot contains several child patches.

- **Properties**:
    - `name`: Name of the plot.
    - `offset`: Spatial vector indicating the plot's position on its parent island.
    - `children`: List of child patches within the plot.
    - (And other properties associated with visual effects, resource spawning and/or creature behaviors)

In [94]:
#
# Patches and offsets, for different Plot layouts
#

PLOT_PATCHES = {
    "medium_vertical": [           # List of patch types inside each plot layout
        {
            "layout": "large_patch",
            "offset": (0,0)
        },
        {
            "layout": "large_patch",
            "offset": (0,3)
        }
    ],
    "medium_horizontal": [
        {
            "layout": "large_patch",
            "offset": (0,0)
        },
        {
            "layout": "large_patch",
            "offset": (3,0)
        }
    ],
    "extra_large_horizontal": [
        {
            "layout": "large_patch",
            "offset": (0,0)
        },
        {
            "layout": "large_patch",
            "offset": (3,0)
        },
        {
            "layout": "large_patch",
            "offset": (6,0)
        }
    ],
    "horizontal": [
        {
            "layout": "small_patch",
            "offset": (0,0)
        },
        {
            "layout": "medium_patch",
            "offset": (2,0)
        },
        {
            "layout": "large_patch",
            "offset": (6,0)
        }
    ],
    "default": [                    # 5 x 3 default
        {
            "layout": "large_patch",
            "offset": (0,0)
        },
        {
            "layout": "medium_patch",
            "offset": (0,3)
        }
    ]
}

class Plot(NestedComponent):
    DEFAULT_LAYOUT = "default"

    def __init__(self, parent_island, plot_data, mods):
        """
        Represents a plot layout within an island containing properties such as shade level and light color.
        """
        super().__init__()  # Initialize parent class
        self.level = "plot"
        self._parent_island = parent_island
        self.mods = mods.get(self.level, {})

        # Initialize plot attributes
        self.name = plot_data.get("name", "Plot")
        self.unique_id = f"{self._parent_island.get_name()} > {self.name}"
        self.offset = plot_data.get("offset", (0,0))
        self.layout = plot_data.get("layout", Plot.DEFAULT_LAYOUT)
        self.pastime_types = plot_data.get("pastime_types", {})

        # Plot-level variables
        self.shade_level = plot_data.get("shade_level", random.choice(["High", "Medium", "Low"]))
        self.light_color = plot_data.get("light_color", random.choice(["Green", "Yellow", "Purple", "Blue"]))

        # Initialize child patches for the plot
        if self.layout not in PLOT_PATCHES:
            raise ValueError(f"Unknown plot layout: {self.layout}")
        self.children = [Patch(self, patch_data, mods) for patch_data in PLOT_PATCHES[self.layout]]

    def get_id(self):
        return self.unique_id

    def get_offset(self):
        return self.offset

    def get_parent_island(self):
        return self._parent_island

    def get_parent_island_name(self):
        return self._parent_island.get_name()

    def get_pastime_types(self):
        return self.pastime_types

    def get_level_variables(self):
        return {
            "name": self.name,
            "shade_level": self.shade_level,
            "light_color": self.light_color
        }

### 1a4. Islands

The `Island` class represents an island layout within a world. Each island has properties and contains several child plots.

- **Properties**:
    - `name`: Name of the island.
    - `offset`: Spatial vector determining the island's position within the world.
    - `children`: List of child plots associated with the island.
  
- **Functions**:
    - `get_name()`: Returns the name of the island.
    - `get_plots()`: Returns all the child plots of the island.
    - `get_plot_ids()`: Returns unique IDs for each plot on the island.
    - `get_plot_with_id()`: Fetches a plot with a specific ID.

In [95]:
#
# Plots are organized into islands
#

# I like X, e.g. "I like Sports!"
ACTIVITY_THEMES = ["Sports", "Art", "Music", "Reading"]

# I am X, e.g. "I am Playing Football!"
ACTIVITIES_BY_THEME = {
    "Sports": ["Playing Football", "Playing Soccer", "Playing Baseball", "Playing Tennis"],
    "Art": ["Painting", "Sketching", "Writing", "Sculpting"],
    "Music": ["Tap Dancing", "Playing Guitar", "Playing Drums", "Playing Piano"],
    "Reading": ["Reading Fiction", "Reading Comic Books", "Reading Magazines", "Reading Poetry"]
}


# During Activities, one possible plot (determined by activity choice): 4 per island
# During Rest, on home island one of multiple plots (5 per island?)
# During Hangout, by preferred island (4 per island)
# During Tend, by preferred island (all plots)

# P1: tend, activity1, rest
# P2: tend, activity2
# P3: tend, activity3, rest
# P4: tend, activity4
# P5: tend, hangout, rest
# P6: tend, hangout
# P7: tend, hangout, rest
# P8: tend, hangout
# P9: tend, rest

NINE_PLOTS = [
        { # ACTIVITY
            "name": "Secret Forest",
            "offset": (12, 30),
            "layout": "extra_large_horizontal",
            "pastime_types": {
                "hangout": False,
                "tend": True,
                "activity": True, #1: Reading
                "rest": True #1
            }
        },
        { # ACTIVITY
            "name": "North Fields",
            "offset": (24, 27),
            "layout": "medium_horizontal",
            "pastime_types": {
                "hangout": False,
                "tend": True,
                "activity": True, #2 Sports
                "rest": False
            }
        },
        { # ACTIVITY
            "name": "Community Garden",
            "offset": (15, 9),
            "pastime_types": {
                "hangout": False,
                "tend": True,
                "activity": True, #3: Art
                "rest": True #2
            }
        },
        { # ACTIVITY
            "name": "Musical Mines",
            "offset": (26, 6),
            "pastime_types": {
                "hangout": False,
                "tend": True,
                "activity": True, #4: Music
                "rest": False
            }
        },
        {
            "name": "Large Garden",
            "offset": (0, 24),
            "shade_level": 75,
            "light_color": 75,
            "layout": "medium_vertical",
            "pastime_types": {
                "hangout": True, #1
                "tend": True,
                "activity": False,
                "rest": True #3
            }
        },
        {
            "name": "South Fields",
            "offset": (27, 18),
            "pastime_types": {
                "hangout": True, #2
                "tend": True,
                "activity": False,
                "rest": False
            }
        },
        {
            "name": "Small Clearing",
            "offset": (0, 15),
            "layout": "horizontal",
            "pastime_types": {
                "hangout": True, #3
                "tend": True,
                "activity": False,
                "rest": True #4
            }
        },
        {
            "name": "Small Garden",
            "offset": (12, 20),
            "layout": "medium_horizontal",
            "pastime_types": {
                "hangout": True, #4
                "tend": True,
                "activity": False,
                "rest": False
            }
        },
        {
            "name": "Large Clearing",
            "offset": (12, 0),
            "layout": "extra_large_horizontal",
            "pastime_types": {
                "hangout": False,
                "tend": True,
                "activity": False,
                "rest": True #5
            }
        }
    ]

ISLAND_PLOTS = {
    'Growth': copy.deepcopy(NINE_PLOTS),
    'Stability': copy.deepcopy(NINE_PLOTS),
    'Light': copy.deepcopy(NINE_PLOTS),
    'Shadow': copy.deepcopy(NINE_PLOTS)
}

islands_list = list(ISLAND_PLOTS.keys())
for island in islands_list:
    for plot in ISLAND_PLOTS[island]:
        if plot["name"] == "Musical Mines":
            activity = ACTIVITIES_BY_THEME["Music"][islands_list.index(island)]
            plot["pastime_types"]["activity"] = { activity : True }
        if plot["name"] == "Secret Forest":
            activity = ACTIVITIES_BY_THEME["Reading"][islands_list.index(island)]
            plot["pastime_types"]["activity"] = { activity : True }
        if plot["name"] == "North Fields":
            activity = ACTIVITIES_BY_THEME["Sports"][islands_list.index(island)]
            plot["pastime_types"]["activity"] = { activity : True }
        if plot["name"] == "Community Garden":
            activity = ACTIVITIES_BY_THEME["Art"][islands_list.index(island)]
            plot["pastime_types"]["activity"] = { activity : True }

In [96]:
class Island(NestedComponent):
    DEFAULT_NAME = "Island"

    def __init__(self, parent_world, island_data, mods):
        """
        Represents an island layout within a world containing properties and child plots.
        """
        super().__init__()  # Initialize parent class
        self.level = "island"
        self.parent_world = parent_world
        self.mods = mods.get(self.level, {})

        # Initialize island attributes
        self.name = island_data.get("name", Island.DEFAULT_NAME)
        self.offset = island_data.get("offset", (0, 0))
        self.children = [Plot(self, plot_data, mods) for plot_data in ISLAND_PLOTS.get(self.name, [])]

    def get_name(self):
        return self.name

    def get_plots(self):
        return self.children

    def get_plot_ids(self):
        return [plot.get_unique_id() for plot in self.children]

    def get_plot_with_id(self, id):
        for plot in self.children:
            if plot.get_unique_id() == id:
                return plot
        return None

    def get_level_variables(self):
        return {
            "island_name": self.name
        }

### 1a5. World

The `World` class represents the highest level of organization in our game, tying together individual islands, tracking world-related elements like time (`clock`) and `weather`, and ensuring updates are propagated appropriately.

- **Properties**:
    - `clock`: A reference to the game's global clock, controlling time progression.
    - `weather`: Represents the game's dynamic weather system.
    - `children`: A list of child islands encompassed by the world.

- **Functions**:
    - `get_weather()`: Provides a snapshot of the current weather conditions.
    - `get_datetime()`: Returns the current day and time of day.
    - `get_island_by_name(island_name)`: Retrieves an island object by its name.
    - `get_locations()`: Lists all available locations within the world, including islands, plots, and their respective attributes.
    - `get_level_variables()`: Provides a summary of world-specific variables like date, time, season, and weather conditions.
    - `next()`: Advances the game's time by one unit, updates the weather if necessary, and ensures all underlying systems and elements are in sync.

In [97]:
#
# Islands are organized into a World
#

# Island data
ISLAND_WIDTH = 40
ISLAND_HEIGHT = 40
ISLANDS = [
    {
        "name": "Growth",
        "offset": (0, 0)
    },
    {
        "name": "Shadow",
        "offset": (-ISLAND_WIDTH, 0),
    },
    {
        "name": "Stability",
        "offset": (-ISLAND_WIDTH, -ISLAND_HEIGHT)
    },
    {
        "name": "Light",
        "offset": (0, -ISLAND_HEIGHT)
    }
]

class World(NestedComponent):
    """
    Represents the game's world, containing a collection of islands, time, and weather conditions.
    """
    def __init__(self, clock, weather, islands_data, mods, base_weights):
        super().__init__()  # Initialize parent class
        self.level = "world"

        self.clock = clock
        self.weather = weather

        # Create Islands and pass mods down
        self.mods = mods.get(self.level, {})
        self.children = [Island(self, island, mods) for island in islands_data]

        # Initialize properties and propagate initial weights
        locations = self.get_locations()
        self.update_from_base_weights(base_weights)

    def get_weather(self):
        return {
            "rain_condition": self.weather.get_rain_condition(),
            "temp": self.weather.get_temp()
        }

    def get_datetime(self):
        return {
            "day": self.clock.get_day(),
            "time_of_day": self.clock.get_time_of_day()
        }

    def get_island_by_name(self, island_name):
        matching_islands = [child for child in self.children if child.get_name() == island_name]
        return matching_islands[0] if matching_islands else None

    def get_locations(self):
        locations = []
        for island in self.children:
            plots = island.get_plots()
            for plot in plots:
                locations.append({"island": island.get_name(), "pastime_types": plot.get_pastime_types(), "plot_id": plot.get_id(), "x": plot.get_offset()[0], "y": plot.get_offset()[1], "weight": 1})
        return locations

    def get_level_variables(self):
        return {
            "day": self.clock.get_day(),
            "time_of_day": self.clock.get_time_of_day(),
            "day_of_week": self.clock.get_day_of_week(),
            "season": self.clock.get_season(),
            "rain_condition": self.weather.get_rain_condition(),
            "temp": self.weather.get_temp(),
        }

    def next(self):
        self.clock.next()
        if self.clock.get_time_of_day() == "AM Early":
            self.weather.next()

        # After updating the clock and weather, reset weights, reapply mods and propagate changes
        self.update_from_base_weights(self.base_weights)

# 2. Creatures and Factions

## **The Creature Population and Factions**

### Factions Overview

  - The game has four factions of Creatures: **Light**, **Shadow**, **Growth**, and **Stability**.
  - Each faction has a different visual appearance. Light creatures look like mushrooms; Shadow like bugs; Growth like trees; Stability like rocks.
  - Factions tend to share patterns of behaviors, respond to items in a similar way, and are more likely to be found on their home island.

- **Opposite and Synergistic Factions**
  - Light/Shadow and Growth/Stability are **Opposite Factions**. Light/Growth and Shadow/Stability are **Synergistic Factions**.
  - These faction pair relationships are used in **Crafting Logic** to determine the `faction_blend` of crafted items, which can influence the effects the items have on creatures.
  - These relationships are also used in determining imbalances in gameplay, with imbalances in the relative faction stats of Light/Shadow and Growth/Stability determining probabilities of **Imbalance Events**.

- **Faction Members**:
  - Each faction has **25 Creatures**: 5 of each `creature_style` and 5 of each `creature_color`, for 25 unique Creatures per Faction (one of each unqiue color/type combination).
  - The 5 Creature Colors are: Blue, Orange, Red, Green, and Purple.
  - The 5 Creature Styles are: A, B, C, D, and E. These correspond to Body Type art variations for each Faction.
  - Each creature has a unique `creature_id` and `creature_name`.

- **Creature Behaviors and Interactions**
  - Each time of day, creatures move to map plots probabilitistically, in patterns influenced by their faction, individual variables, and stats. As an example, Light type creatures tend to 'rest' at night, and will most likely be found on their home island, on plots that have pastime_type 'rest'. These are described in the **Creature Behaviors** section.
  - Additional creature variables used in updating creature states or behaviors are: `friendship_threshold` and `creature_profile`, used in the creature friendship and item effects logic (see **Item Effects**).

---

#### **Creature and Faction Stats**

- **Overview**:
  - Each individual creature has **mood**, **health**, and **social** stats that evolve over time and in response to gameplay dynamics.

- **Mood & Health**:
  - The `mood` and `health` metrics are integer values between **1-100**, representing the creature's happiness and health level, respectively.
  
- **Social**:
  - The creature's `social` stat is calculated from the creature's relationship levels with every player. The `get_social` method in the Creature class below illustrates the stat calculation, which is currently just an average over all player relationship levels.

- **Faction Stats**
  - Each faction has faction stats, comprising a `faction_health`, `faction_mood` and `faction_social` that are the sum over all of the member creatures' health, mood and social stats, respectively.

- **Imbalances in Stats**
  - An Imbalance Event can be caused when there is an imbalance in the health, social, or mood stats of two opposing factions. The simplest version of this for now is to set a threshold difference between two faction stats, e.g. between the `faction_health` values for Light and Shadow factions.
  - Moderate imbalances can trigger a Newspaper "headline" alerting the players to an imbalance. Ex: "Stability creatures thriving, while Growth creatures remain sick." See: **Newspaper** section.

---

#### **Player Relationships to Creatures and Factions**

- **Relationship Levels**:
  - Creatures have a relationship level with each indivdidual player in the game, stored as an integer ranging from **1-100**.
  
- **Friendship**:
  - If a creature's relationship level with a specific player is above a threshold value, the player and creature are considered **"friends"**.
  - Each creature has a `friendship_threshold`, making it more or less challenging to befriend some creatures or creature types. Most likely this threshold is associated with the `creature_style`, e.g. "Growth Style A creatures are very friendly; Growth Style E are hard to befriend."

---

#### **Connections to Other Game Systems**

- **Altering Creature Stats with Items**:
  - Creature stats can be altered by items such as **potions**, **gifts**, or **foods**. The effects of items on creatures are described in the **Item Effects** section.

- **Creature Stats influence Item Effects**:
  - Creature stats can influence their "preferences" for specific items (can modify item effects). Ex: a creature with a low health stat may not have their mood improved by gifts until their health improves. See the **Item Effects** section.

- **Nighttime Update Logic**:
  - The creature stats might increase or decrease based on other stats, Faction stats, Island Health, etc as specified by the **Nighttime Update Logic**.

- **Map Access and Creature Friendships**:
  - Players may gain access to a special map plot by becoming friends with creatures, groups of creatures, or a threshold number of creatures within a faction.

### **Population**

In [98]:
FACTIONS = ["Growth", "Stability", "Shadow", "Light"]

BALANCED_FACTION_COMBINATIONS = [("Light", "Shadow"), ("Growth", "Stability")]
SYNERGISTIC_FACTION_COMBINATIONS = [("Light", "Growth"), ("Shadow", "Stability")]
NEUTRAL_FACTION_COMBINATIONS = [("Light", "Stability"), ("Shadow", "Growth")]

CREATURE_STYLES = ["A", "B", "C", "D", "E"]
CREATURE_COLORS = ["Blue", "Orange", "Red", "Green", "Purple"]
DEFAULT_FRIENDSHIP_THRESHOLD = 50

class CreaturePopulation(NestedComponent):
    def __init__(self, world, creatures_data, mods, CREATURE_CHOICES):
        super().__init__()  # Initialize parent class
        self.level = "population"

        self.mods = mods.get(self.level, {})
        self.children = [Faction(world, faction_name, creatures_data[faction_name], mods) for faction_name in creatures_data.keys()]
        self.creature_locations = {}

        # Set initial weights and propagate
        self.base_weights = CREATURE_CHOICES
        self.base_weights["locations"] = world.get_locations()
        self.update_from_base_weights(self.base_weights)

    def get_creatures_list(self):
        all_creatures = []
        for faction in self.children:
            all_creatures.extend(faction.get_creatures_list())  # Changed from append to extend
        return all_creatures

    def get_creature_locations(self):
        return self.creature_locations

    def get_faction_by_name(self, faction_name):
        for faction in self.children:
            if faction.name == faction_name:
                return faction
        return None

    def get_faction_stats(self, faction_name):
        faction = self.get_faction_by_name(faction_name)
        if faction:
            return faction.get_stats()
        return {}

    def get_all_faction_stats(self):
        faction_stats = {}
        for faction in self.children:
            faction_stats[faction.name] = faction.get_stats()
        return faction_stats

    def get_level_variables(self):
        return {
            "population_stats": self.get_all_faction_stats()
        }

    def next(self, world, creature_data):
        all_locations = {}
        for faction in self.children:
            faction_locations = faction.next(world, creature_data)

            for plot_id, creatures_in_plot in faction_locations.items():
                all_locations.setdefault(plot_id, []).extend(creatures_in_plot)

        self.creature_locations = all_locations
        return all_locations

### **Faction**

In [99]:
class Faction(NestedComponent):
    def __init__(self, world, faction_name, faction_creatures, mods):
        super().__init__()  # Initialize parent class
        self.level = "faction"
        self.world = world
        self.mods = mods.get(self.level, {})

        self.name = faction_name
        self.children = [Creature(world, self, creature_data, mods) for creature_data in faction_creatures]

    def get_name(self):
        return self.name

    def get_creatures_list(self):
        all_creatures = []
        for creature in self.children:
            all_creatures.append(creature.get_name())
        return all_creatures

    def is_friends(self, player_name):
        return self.player_relationship_levels[player_name] > DEFAULT_FRIENDSHIP_THRESHOLD

    def get_stats(self):
        total_mood = sum([creature.get_mood() for creature in self.children])
        total_health = sum([creature.get_health() for creature in self.children])
        total_social = sum([creature.get_social() for creature in self.children])

        return {
            'mood': total_mood,
            'health': total_health,
            'social': total_social
        }

    def get_level_variables(self):
        vars = {
            "faction_name": self.name,
            "faction_stats": self.get_stats(),
            "time_of_day": self.world.clock.get_time_of_day()
        }

        return vars

    def next(self, world, creature_data):
        # re-calculate weights given new world state (time_of_day)
        self.world = world
        self.update_from_base_weights(self.base_weights)

        locations = {}
        for creature in self.children:
            plot_id, island_name = creature.next(world, creature_data)
            locations.setdefault(plot_id, []).append(creature)
        return locations

### **Creature**

In [100]:
class Creature(NestedComponent):
    def __init__(self, world, faction, creature_data, mods):
        super().__init__()  # Initialize parent class
        self.level = "creature"
        self.mods = mods.get(self.level, {})

        self.faction = faction
        self.name = creature_data.get("creature_name", "Creature Name")
        self.color = creature_data.get("creature_color", random.choice(CREATURE_COLORS))
        self.style = creature_data.get("creature_style", random.choice(CREATURE_STYLES))

        self.pastime = "activity"
        self.activity = "Reading Fiction"
        self.world = world
        self.target_location = creature_data.get("location", random.choice(ISLAND_PLOTS[faction.get_name()]))

        self.health = creature_data.get("health", 50)
        self.mood = creature_data.get("mood", 50)
        self.player_relationship_levels = {}
        self.friendship_threshold = creature_data.get("friendship_threshold", DEFAULT_FRIENDSHIP_THRESHOLD)

    ## CREATURE INFO
    def get_name(self):
        return self.name

    def get_color(self):
        return self.color

    def get_attribute(self):
        return self.attribute

    def get_health(self):
        return self.health

    def get_mood(self):
        return self.mood

    def get_social(self):
        return sum(self.player_relationship_levels.values())

    def get_stats(self):
        return {
            'mood': self.mood,
            'health': self.health,
            'social': self.get_social()
        }

    def get_level_variables(self):
        return {
            "creature_faction": self.faction,
            "creature_name": self.name,
            "creature_style": self.style,
            "creature_color": self.color,
            "pastime": self.pastime,
            "activity": self.activity
        }

    def get_num_friends(self):
        return sum(1 for value in self.player_relationship_levels.values() if value > self.friendship_threshold)

    def is_friends(self, player_name):
        return self.player_relationship_levels.get(player_name, 0) > self.friendship_threshold

    def set_player_relationship_levels(self, player_name, level):
        self.player_relationship_levels[player_name] = level

    ## CREATURE BEHAVIORS AND INTERACTIONS
    def give_item(self, player, item):
        item_category = item["item_category"]
        base_effects = ITEM_BASE_EFFECTS[item_category]

        effects = base_effects
        self.apply_stats_effects(player, effects)
        return effects

    def apply_stats_effects(self, player, effects):
        self.health = self.health + effects["health"]
        self.mood = self.mood + effects["mood"]
        player_name = player.get_name()
        self.player_relationship_levels[player_name] = self.player_relationship_levels.get(player_name, 50) + effects["social"]

    def next(self, world, creature_data):
        # update variables and re-calculate weights
        self.world = world
        self.update_from_base_weights(self.base_weights)

        # Select a pastime
        pastimes = self.weights["pastimes"]
        weights = [pastime["weight"] for pastime in pastimes]
        self.pastime = random.choices(pastimes, weights, k=1)[0]["pastime"]
        self.update_from_base_weights(self.base_weights)

        locations = self.weights["locations"]
        self.activity = ""

        # Pick an activity if needed, and find which plots are allowed for this pastime and/or activity
        if self.pastime == "activity":
            # Select activity
            activities = self.weights["activities"]
            weights = [activity["weight"] for activity in activities]
            self.activity = random.choices(activities, weights, k=1)[0]
            self.update_from_base_weights(self.base_weights)

            allowed_plots = []
            for plot in locations:
                if plot["pastime_types"]["activity"]:
                    check = plot["pastime_types"]["activity"].get(self.activity["activity_name"], {})
                    if check:
                        allowed_plots.append(plot)

        elif self.pastime == "rest":
            # Filter locations based on the pastime condition
            allowed_plots = [location for location in locations if location["island"] == self.faction.get_name()]

        else:
            # Filter locations based on the pastime condition
            allowed_plots = [location for location in locations if location["pastime_types"].get(self.pastime, False)]

        # Pick a plot from the allowed plots
        weights = [location["weight"] for location in allowed_plots]
        plot = random.choices(allowed_plots, weights, k=1)[0]
        self.target_location_id = plot["plot_id"]

        base_world_data = {
            "time": self.world.clock.get_current_time()
        }

        base_creature_data = {
            "creature_name": self.name,
            "creature_faction": self.faction.get_name(),
            "creature_style": self.style,
            "creature_color": self.color
        }

        base_migration_data = {
            "pastime": self.pastime,
            "plot_id": self.target_location_id,
            "x": plot["x"],
            "y": plot["y"],
            "island": plot["island"]
        }

        if self.activity:
            migration_data = {
                **base_world_data,
                **base_creature_data,
                **base_migration_data,
                "activity_theme": self.activity["activity_theme"],
                "activity": self.activity["activity_name"]
            }
        else:
            migration_data = {
                **base_world_data,
                **base_creature_data,
                **base_migration_data
            }

        creature_data["migrations"].append(migration_data)

        creature_data["stats"].append({
            **base_world_data,
            **base_creature_data,
            "creature_health": self.health,
            "creature_mood": self.mood,
            "creature_social": self.get_social()
        })

        return self.target_location_id, plot["island"]

## **Creatures List**

In [101]:
CREATURES = {
"Light": [
    {"creature_name": "Glint", "creature_color": "Blue", "creature_style": "A"},
    {"creature_name": "Prismaros", "creature_color": "Blue", "creature_style": "B"},
    {"creature_name": "Aurora", "creature_color": "Blue", "creature_style": "C"},
    {"creature_name": "Celestia", "creature_color": "Blue", "creature_style": "D"},
    {"creature_name": "Radiant", "creature_color": "Blue", "creature_style": "E"},
    {"creature_name": "Amethyst", "creature_color": "Purple", "creature_style": "A"},
    {"creature_name": "Luminara", "creature_color": "Purple", "creature_style": "B"},
    {"creature_name": "Iris", "creature_color": "Purple", "creature_style": "C"},
    {"creature_name": "Seraphim", "creature_color": "Purple", "creature_style": "D"},
    {"creature_name": "Violet", "creature_color": "Purple", "creature_style": "E"},
    {"creature_name": "Pyroshimmer", "creature_color": "Red", "creature_style": "A"},
    {"creature_name": "Blaze", "creature_color": "Red", "creature_style": "B"},
    {"creature_name": "Radiance", "creature_color": "Red", "creature_style": "C"},
    {"creature_name": "Ignatia", "creature_color": "Red", "creature_style": "D"},
    {"creature_name": "Phoenix", "creature_color": "Red", "creature_style": "E"},
    {"creature_name": "Viridium", "creature_color": "Green", "creature_style": "A"},
    {"creature_name": "Meadowlark", "creature_color": "Green", "creature_style": "B"},
    {"creature_name": "Lumigrove", "creature_color": "Green", "creature_style": "C"},
    {"creature_name": "Verdant", "creature_color": "Green", "creature_style": "D"},
    {"creature_name": "Zephyr", "creature_color": "Green", "creature_style": "E"},
    {"creature_name": "Solara", "creature_color": "Orange", "creature_style": "A"},
    {"creature_name": "Lumos", "creature_color": "Orange", "creature_style": "B"},
    {"creature_name": "Aurelia", "creature_color": "Orange", "creature_style": "C"},
    {"creature_name": "Luminary", "creature_color": "Orange", "creature_style": "D"},
    {"creature_name": "Radiantwing", "creature_color": "Orange", "creature_style": "E"}],
"Shadow": [
    {"creature_name": "Umbra", "creature_color": "Blue", "creature_style": "A"},
    {"creature_name": "Shade", "creature_color": "Blue", "creature_style": "B"},
    {"creature_name": "Duskray", "creature_color": "Blue", "creature_style": "C"},
    {"creature_name": "Twilight", "creature_color": "Blue", "creature_style": "D"},
    {"creature_name": "Nyx", "creature_color": "Blue", "creature_style": "E"},
    {"creature_name": "Obsidian", "creature_color": "Purple", "creature_style": "A"},
    {"creature_name": "Shadowstrike", "creature_color": "Purple", "creature_style": "B"},
    {"creature_name": "Eclipse", "creature_color": "Purple", "creature_style": "C"},
    {"creature_name": "Nocturna", "creature_color": "Purple", "creature_style": "D"},
    {"creature_name": "Nox", "creature_color": "Purple", "creature_style": "E"},
    {"creature_name": "Pyroshadow", "creature_color": "Red", "creature_style": "A"},
    {"creature_name": "Ember", "creature_color": "Red", "creature_style": "B"},
    {"creature_name": "Shadowfire", "creature_color": "Red", "creature_style": "C"},
    {"creature_name": "Inferno", "creature_color": "Red", "creature_style": "D"},
    {"creature_name": "Ashen", "creature_color": "Red", "creature_style": "E"},
    {"creature_name": "Verdigris", "creature_color": "Green", "creature_style": "A"},
    {"creature_name": "Murk", "creature_color": "Green", "creature_style": "B"},
    {"creature_name": "Shadowthorn", "creature_color": "Green", "creature_style": "C"},
    {"creature_name": "Mossbane", "creature_color": "Green", "creature_style": "D"},
    {"creature_name": "Shadowleaf", "creature_color": "Green", "creature_style": "E"},
    {"creature_name": "Umbertide", "creature_color": "Orange", "creature_style": "A"},
    {"creature_name": "Gloomsong", "creature_color": "Orange", "creature_style": "B"},
    {"creature_name": "Shadowflare", "creature_color": "Orange", "creature_style": "C"},
    {"creature_name": "Twilightspire", "creature_color": "Orange", "creature_style": "D"},
    {"creature_name": "Nightshade", "creature_color": "Orange", "creature_style": "E"}],
"Growth": [
    {"creature_name": "Flora", "creature_color": "Blue", "creature_style": "A"},
    {"creature_name": "Lush", "creature_color": "Blue", "creature_style": "B"},
    {"creature_name": "Sylvan", "creature_color": "Blue", "creature_style": "C"},
    {"creature_name": "Meadow", "creature_color": "Blue", "creature_style": "D"},
    {"creature_name": "Zephyrus", "creature_color": "Blue", "creature_style": "E"},
    {"creature_name": "Blossom", "creature_color": "Purple", "creature_style": "A"},
    {"creature_name": "Petal", "creature_color": "Purple", "creature_style": "B"},
    {"creature_name": "Thorn", "creature_color": "Purple", "creature_style": "C"},
    {"creature_name": "Rose", "creature_color": "Purple", "creature_style": "D"},
    {"creature_name": "Fern", "creature_color": "Purple", "creature_style": "E"},
    {"creature_name": "Mosswhisper", "creature_color": "Red", "creature_style": "A"},
    {"creature_name": "Bramble", "creature_color": "Red", "creature_style": "B"},
    {"creature_name": "Sunleaf", "creature_color": "Red", "creature_style": "C"},
    {"creature_name": "Emberbloom", "creature_color": "Red", "creature_style": "D"},
    {"creature_name": "Chlorophyll", "creature_color": "Red", "creature_style": "E"},
    {"creature_name": "Viridius", "creature_color": "Green", "creature_style": "A"},
    {"creature_name": "Willow", "creature_color": "Green", "creature_style": "B"},
    {"creature_name": "Oakheart", "creature_color": "Green", "creature_style": "C"},
    {"creature_name": "Ivy", "creature_color": "Green", "creature_style": "D"},
    {"creature_name": "Thicket", "creature_color": "Green", "creature_style": "E"},
    {"creature_name": "Solstice", "creature_color": "Orange", "creature_style": "A"},
    {"creature_name": "Gleam", "creature_color": "Orange", "creature_style": "B"},
    {"creature_name": "Radiant Blossom", "creature_color": "Orange", "creature_style": "C"},
    {"creature_name": "Vivaro", "creature_color": "Orange", "creature_style": "D"},
    {"creature_name": "Evergreen", "creature_color": "Orange", "creature_style": "E"}],
"Stability": [
    {"creature_name": "Granite", "creature_color": "Blue", "creature_style": "A"},
    {"creature_name": "Steadfast", "creature_color": "Blue", "creature_style": "B"},
    {"creature_name": "Vanguard", "creature_color": "Blue", "creature_style": "C"},
    {"creature_name": "Shieldstone", "creature_color": "Blue", "creature_style": "D"},
    {"creature_name": "Ironclad", "creature_color": "Blue", "creature_style": "E"},
    {"creature_name": "Ash", "creature_color": "Purple", "creature_style": "A"},
    {"creature_name": "Stonewall", "creature_color": "Purple", "creature_style": "B"},
    {"creature_name": "Firmament", "creature_color": "Purple", "creature_style": "C"},
    {"creature_name": "Grit", "creature_color": "Purple", "creature_style": "D"},
    {"creature_name": "Solidarity", "creature_color": "Purple", "creature_style": "E"},
    {"creature_name": "Ambrose", "creature_color": "Red", "creature_style": "A"},
    {"creature_name": "Hearth", "creature_color": "Red", "creature_style": "B"},
    {"creature_name": "Ironheart", "creature_color": "Red", "creature_style": "C"},
    {"creature_name": "Resolute", "creature_color": "Red", "creature_style": "D"},
    {"creature_name": "Valiant", "creature_color": "Red", "creature_style": "E"},
    {"creature_name": "Verdantia", "creature_color": "Green", "creature_style": "A"},
    {"creature_name": "Stalwart", "creature_color": "Green", "creature_style": "B"},
    {"creature_name": "Strongroot", "creature_color": "Green", "creature_style": "C"},
    {"creature_name": "Solidrock", "creature_color": "Green", "creature_style": "D"},
    {"creature_name": "Endurance", "creature_color": "Green", "creature_style": "E"},
    {"creature_name": "Aurum", "creature_color": "Orange", "creature_style": "A"},
    {"creature_name": "Guardian", "creature_color": "Orange", "creature_style": "B"},
    {"creature_name": "Ironcliff", "creature_color": "Orange", "creature_style": "C"},
    {"creature_name": "Steelsong", "creature_color": "Orange", "creature_style": "D"},
    {"creature_name": "Lastingheart", "creature_color": "Orange", "creature_style": "E"}],
}

## **Creature Behaviors**

### **Creatures Choose Pastimes and Locations**

At any time of day, a creature selects a particular "pastime", which determines their movements to specific map locations, and can modify item effects.

There are four pastimes:

- **Rest**: This represents the creatures' "sleep" time. The creature preferentially moves to a plot on their home island that is marked as a place for 'rest'. If animation is affected, creatures appear be sitting still or asleep. Health potion effects are boosted.
- **Hangout**: This pastime represents creatures 'socializing' or gathering together. The creature moves to a plot, preferring their home island and synergistic islands, that is marked as a place for 'hangout'. If animation is affected, the creatures doing 'hangout' on the same plot may stand nearby each other or around a specific tile area. Certain Food item effects may be boosted, e.g. "Growth creatures prefer Ginger Cookies at their hangouts." These can be hinted at in the **Newspaper**, e.g. "Cookie Day for the Light creatures".
- **Activity**: The creature goes to a plot on any island that is marked as a place for 'activity'. A specific themed activity is chosen based on creature preferences. Gift effects are boosted for that theme.
- **Tend**: The creature goes to plots on any island that are marked as a place for 'tend'.

**Patterns and Effects**:
- Different factions have different pasttime patterns depending on time of day.
- Factions prefer their own island, then synergistic islands.

**Data**: Logs of player interactions with creatures record:
- the `day` and `time_of_day`
- the location of the interaction (by `plot_id`)
- the `creature_id`
- the creature's current `creature_pastime`
- any gifting events, if the player gives the creature an item

**Open Questions**:
1. To what extent do these creature pastimes affect visual appearance of creatures, through either art or animation? Do creatures need to appear to be sleeping during 'rest'?
2. Should 'rest' mean the player can not interact with the creature at that time, i.e. that the creature is 'asleep'?

#### **Pattern: Factions tend to select different pastimes at different times of day**

In [102]:
creature_effects = {}

creature_mods = {}
creature_mods["faction"] = []
creature_mods["creature"] = []

# Creatures select a pasttime then select a plot for it
CREATURE_PASTIME_CHOICES = [
    {"pastime": "hangout", "weight": 1},  # go to "hangout" plot (by body_type), food effects boosted
    {"pastime": "rest", "weight": 1},     # go to "rest" plot (by body_type), health_potion effects boosted
    {"pastime": "activity", "weight": 1}, # go to "activity" plot (by body_type), gift effects boosted
    {"pastime": "tend", "weight": 1}      # go to "garden" plot (by body_type), gift effects boosted
]
CREATURE_CHOICES = {"pastimes": CREATURE_PASTIME_CHOICES}

#
# Faction-level effects
#

# Different factions have different pasttime patterns depending on time of day
# Currently these are not probabilistic, just determined directly (multipliers 0 on all but one pastime type)
creature_effects["time_of_day"] = {}
creature_effects["time_of_day"]["faction_name"] = {}
creature_effects["time_of_day"]["faction_name"]["pastime"] = {
    'AM Early': {
        'Light': { "hangout": 1, "rest": 0, "activity": 0, "tend": 0 },
        'Shadow': { "hangout": 0, "rest": 1, "activity": 0, "tend": 0 },
        'Growth': { "hangout": 0, "rest": 0, "activity": 1, "tend": 0 },
        'Stability': { "hangout": 0, "rest": 0, "activity": 0, "tend": 1 }
    },
    'AM Late': {
        'Light': { "hangout": 0, "rest": 0, "activity": 0, "tend": 1 },
        'Shadow': { "hangout": 1, "rest": 0, "activity": 0, "tend": 0 },
        'Growth': { "hangout": 0, "rest": 1, "activity": 0, "tend": 0 },
        'Stability': { "hangout": 0, "rest": 0, "activity": 1, "tend": 0 }
    },
    'PM Early': {
        'Light': { "hangout": 0, "rest": 0, "activity": 1, "tend": 0 },
        'Shadow': { "hangout": 0, "rest": 0, "activity": 0, "tend": 1 },
        'Growth': { "hangout": 1, "rest": 0, "activity": 0, "tend": 0 },
        'Stability': { "hangout": 0, "rest": 1, "activity": 0, "tend": 0 }
    },
    'PM Late': {
        'Light': { "hangout": 0, "rest": 1, "activity": 0, "tend": 0 },
        'Shadow': { "hangout": 0, "rest": 0, "activity": 1, "tend": 0 },
        'Growth': { "hangout": 0, "rest": 0, "activity": 0, "tend": 1 },
        'Stability': { "hangout": 1, "rest": 0, "activity": 0, "tend": 0 }
    }
}

# Pastime choice depends on time_of_day and faction
def faction_time_of_day_pastime_effects(base_p, vars):
    choices = copy.deepcopy(base_p)
    pastimes = choices["pastimes"]

    indep_var1 = "time_of_day"
    indep_var2 = "faction_name"

    prop = "pastime"
    current_val1 = str(vars.get(indep_var1, ""))
    current_val2 = str(vars.get(indep_var2, ""))

    if current_val1 and current_val2:
        effects_to_apply = creature_effects.get(indep_var1, {}).get(indep_var2, {}).get(prop, {})
        current_multipliers = effects_to_apply.get(current_val1, {}).get(current_val2, {})
        for pastime in pastimes:
            for key, multiplier in current_multipliers.items():
                if pastime[prop] == key:
                    pastime["weight"] = pastime["weight"] * multiplier
    return choices

creature_mods["faction"].append(faction_time_of_day_pastime_effects)

#### **Pattern: For "Hangout" and "Tend" Pastimes, factions prefer their own islands, then synergistic islands**

In [103]:
# factions generally select locations on their own islands, then prefer locations on their synergistic faction's island
def faction_island_preference(base_p, vars):
    choices = copy.deepcopy(base_p)
    locations = choices["locations"]

    SAME_FACTION_MULTIPLIER = 3
    SYNERGISTIC_FACTION_MULTIPLIER = 1.5

    for location in locations:
        if location["island"] == vars["faction_name"]:
            location["weight"] = location["weight"] * SAME_FACTION_MULTIPLIER

        # if synergistic faction
        SYNERGISTIC_FACTION_COMBINATIONS = [("Light", "Growth"), ("Shadow", "Stability")]

        if (location["island"], vars["faction_name"]) in SYNERGISTIC_FACTION_COMBINATIONS:
            location["weight"] = location["weight"] * SYNERGISTIC_FACTION_MULTIPLIER

    return choices;

creature_mods["faction"].append(faction_island_preference)

### **For "Activity" Pastime, Creatures Choose an Activity Theme and Type**

When creatures do an "activity" pastime, they choose a specific activity theme and type according to their preferences. This choice will determine which map plot they go to, and may add boosts to the effects of items gifted to them at that time and plot location.

Themes and types will be decided through co-design with playtesters. Currently as stand-ins, there are four Activity themes, each with four types:

- Sports (Football, Baseball, Tennis, Soccer)
- Music (Tap Dance, Guitar, Drums, Piano)
- Art (Painting, Sketching, Ink, Clay)
- Books (Comic, Pop-up, Story, Coloring)

Each of these 16 activity types corresponds to a plot somewhere on the island that the creature goes to once they select that activity.

#### **Factions have activity theme preferences**

In [104]:
# List choices and give a weight to each choice
ACTIVITY_CHOICES = []
for theme, activities in ACTIVITIES_BY_THEME.items():
    for activity in activities:
        ACTIVITY_CHOICES.append({
            "activity_name": activity,
            "activity_theme": theme,
            "weight": 1
        })
CREATURE_CHOICES["activities"] = ACTIVITY_CHOICES

# Defines multipliers on the probabilities of selecting a particular activity type
creature_effects = {}
creature_effects["faction_name"] = {}
creature_effects["faction_name"]["activity_theme"] = {
    'Light': {
        'Sports': 2
    },
    'Shadow': {
        'Art': 2
    },
    'Growth': {
        'Music': 2
    },
    'Stability': {
        'Reading': 2
    }
}

# Factions have preferred activities
def faction_activity_theme_preference(base_p, vars):
    choices = copy.deepcopy(base_p)
    activities = choices["activities"]

    indep_var = "faction_name"
    prop = "activity_theme"

    current_val = str(vars[indep_var])
    current_multipliers = creature_effects.get(indep_var, {}).get(prop, {}).get(current_val, {})

    for activity in activities:
        for key, value in current_multipliers.items():
            if activity[prop] == key:
                activity["weight"] = activity["weight"] * value
    return choices

creature_mods["faction"].append(faction_activity_theme_preference)

#### **Creature styles have activity theme preferences**

In [105]:
creature_effects["creature_style"] = {}
creature_effects["creature_style"]["activity_theme"] = {
    'A': {
        'Sports': 4,
        'Art': 3,
        'Music': 2,
        'Reading': 1
    },
    'B': {
        'Sports': 3,
        'Art': 4,
        'Music': 1,
        'Reading': 2
    },
    'C': {
        'Sports': 1,
        'Art': 2,
        'Music': 4,
        'Reading': 3
    },
    'D': {
        'Sports': 2,
        'Art': 1,
        'Music': 3,
        'Reading': 4
    },
    'E': {
        'Sports': 1,
        'Art': 1,
        'Music': 1,
        'Reading': 1
    }
}

def creature_style_activity_preference(base_p, vars):
    choices = copy.deepcopy(base_p)
    activities = choices["activities"]

    indep_var = "creature_style"
    prop = "activity_name"

    current_val = str(vars[indep_var])
    current_multipliers = creature_effects.get(indep_var, {}).get(prop, {}).get(current_val, {})

    for activity in activities:
        for key, value in current_multipliers.items():
            if activity[prop] == key:
                activity["weight"] = activity["weight"] * value
    return choices

creature_mods["creature"].append(creature_style_activity_preference)

#### **Creature colors have activity type preferences**

In [106]:
creature_effects["creature_color"] = {}
creature_effects["creature_color"]["activity_name"] = {
    "Light" : {
        'Blue': {
            'Playing Baseball': 3
        },
        'Orange': {
            'Playing Football': 3
        },
        'Red': {
            'Playing Soccer': 3
        },
        'Green': {
            'Playing Tennis': 3
        },
        'Purple': {}
    },
    "Shadow" : {
        'Blue': {
            'Painting': 2
        },
        'Orange': {
            'Sketching': 2
        },
        'Red': {
            'Writing': 2
        },
        'Green': {
            'Sculpting': 2
        },
        'Purple': {}
    },
    "Growth" : {
        'Blue': {
            'Tap Dancing': 2
        },
        'Orange': {
            'Playing Guitar': 2
        },
        'Red': {
            'Playing Piano': 2
        },
        'Green': {},
        'Purple': {
            'Playing Drums': 2
        }
    },
    "Stability" : {
        'Blue': {},
        'Orange': {
            'Reading Fiction': 2
        },
        'Red': {
            'Reading Comic Books': 2
        },
        'Green': {
            'Reading Magazines': 2
        },
        'Purple': {
            'Reading Poetry': 2
        }
    }
}

def creature_style_activity_preference(base_p, vars):
    choices = copy.deepcopy(base_p)
    activities = choices["activities"]

    indep_var1 = "creature_faction"
    indep_var2 = "creature_color"
    prop = "activity_name"

    current_val1 = str(vars[indep_var1])
    current_val2 = str(vars[indep_var2])
    current_multipliers = creature_effects.get(indep_var1, {}).get(indep_var2, {}).get(prop, {}).get(current_val1, {}).get(current_val2, {})

    for activity in activities:
        for key, value in current_multipliers.items():
            if activity[prop] == key:
                activity["weight"] = activity["weight"] * value
    return choices

creature_mods["creature"].append(creature_style_activity_preference)

### **Creatures Make Shop Requests**

Creatures send requests to the shop for specific items and offer trades.

Danny is working on this system.

# 3. Resources and Spawning

## 3a. **Resources**

### Overview

In the game, players can collect 16 different types of resources. These resources spawn across the four islands and are each associated with one of four different factions. Every faction has a unique resource for each of the following categories:

1. Fruit
2. Sweet
3. Magic
4. Material

These resource categories are combined to craft different categories of item: foods, gifts, health potions and growth potions. The specific item type crafted depends on the specific resource types used, while the quality of the item will depend on the qualities of the resources used.

---

### Resource Type Properties
Each resource type has a:
- **Name**: Unique text representing the resource name, e.g. "Stonefruit."
- **Faction**: The faction to which the resource belongs (Growth, Stability, Light, Shadow).
- **Category**: One of four categories of resource (Fruit, Sweet, Magic, Material). These categories are used in crafting logic.
- **Commonness**: A numerical value representing the relative probability of that resource spawning. Higher numbers indicate greater probability.
- **Value**: The resource's value, currently the inverse of "commonness" and may not be necessary. Current use is effectively as a "max quality" for spawned resource instances.
- **Spawn Behavior**: Currently, resources may have one of four "spawn behaviors": plant, bug, mushroom, rock. These can be used to modify spawn probabilities based on variables like `time_of_day` (e.g. "bugs come out in the morning") in a way separated from resource faction, or based on spawn node types if there are multiple (e.g. "bugs are found by overturning rocks"). Depending on eventual resource categories and node types defined, this variable may be removed.
- **Shade Preference**: Additional variables might be used to specify how individual resources interact with map variables in determining spawning probabilities. Currently, `shade_preference` is specific to each resource at the individual level, and can be used in the spawning logic to alter the type or quality of resource spawned.

Note that a higher "commonness" currently corresponds to a proportionally lower "value," so these variables are somewhat redundant.

---

### Spawned Resource Instances
When spawned, each resource has:
- **Unique ID**: Unique id representing the specific resource. Used for tracking resources.
- **Resource Type**: One of the 16 resource types, e.g. "Stonefruit."
- **Resource Quality**: The quality of the spawned resource can vary, with a maximum value set by the value of the resource type. Rare (low `commonnness`) resource types have higher value, so have higher max `resource_quality`.

### Resource Type Definitions

#### Balancing

These resource commonness values are currently balanced to add up to the same number (16) for each faction, with a ratio of 3 : 2 : 2 : 1 for magic: fruit : sweet : material, reflecting the ratios in which they appear in recipes outlined in the **Crafting Logic** section.


In [107]:
RESOURCE_TYPES = [
    {'name': 'Dawnspore', 'faction': 'Light', 'spawn_behavior': 'mushrooms', 'shade_preference': 'Low', 'category': 'fruit', 'commonness': 3, 'value': 60},
    {'name': 'Lumina Dew', 'faction': 'Light', 'spawn_behavior': 'mushrooms', 'shade_preference': 'Low', 'category': 'sweet', 'commonness': 6, 'value': 30},
    {'name': 'Radiance Root', 'faction': 'Light', 'spawn_behavior': 'mushrooms', 'shade_preference': 'Low', 'category': 'magic', 'commonness': 6, 'value': 30},
    {'name': 'Fungus Fiber', 'faction': 'Light', 'spawn_behavior': 'mushrooms', 'shade_preference': 'Low', 'category': 'material', 'commonness': 1, 'value': 180},

    {'name': 'Duskworm Fruit', 'faction': 'Shadow', 'spawn_behavior': 'bugs', 'shade_preference': 'High', 'category': 'fruit', 'commonness': 4, 'value': 45},
    {'name': 'Shadowbug Nectar', 'faction': 'Shadow', 'spawn_behavior': 'bugs', 'shade_preference': 'High', 'category': 'sweet', 'commonness': 2, 'value': 90},
    {'name': 'Beetle Wing', 'faction': 'Shadow', 'spawn_behavior': 'bugs', 'shade_preference': 'High', 'category': 'magic', 'commonness': 9, 'value': 20},
    {'name': 'Dark Silk', 'faction': 'Shadow', 'spawn_behavior': 'bugs', 'shade_preference': 'High', 'category': 'material', 'commonness': 1, 'value': 180},

    {'name': 'Lifeseed', 'faction': 'Growth', 'spawn_behavior': 'plants', 'shade_preference': 'High', 'category': 'fruit', 'commonness': 5, 'value': 36},
    {'name': "Sweetsap", 'faction': 'Growth', 'spawn_behavior': 'plants', 'shade_preference': 'Medium', 'category': 'sweet', 'commonness': 5, 'value': 36},
    {'name': 'Tranquil Moss', 'faction': 'Growth', 'spawn_behavior': 'plants', 'shade_preference': 'Medium', 'category': 'magic', 'commonness': 4, 'value': 45},
    {'name': 'Heartwood', 'faction': 'Growth', 'spawn_behavior': 'plants', 'shade_preference': 'Low', 'category': 'material', 'commonness': 2, 'value': 90},

    {'name': 'Stonefruit', 'faction': 'Stability', 'spawn_behavior': 'rocks', 'shade_preference': 'High', 'category': 'fruit', 'commonness': 4, 'value': 45},
    {'name': 'Crystallized Ginger', 'faction': 'Stability', 'spawn_behavior': 'rocks', 'shade_preference': 'Medium', 'category': 'sweet', 'commonness': 3, 'value': 60},
    {'name': 'Runestone Shards', 'faction': 'Stability', 'spawn_behavior': 'rocks', 'shade_preference': 'Medium', 'category': 'magic', 'commonness': 5, 'value': 36},
    {'name': 'Bedrock', 'faction': 'Stability', 'spawn_behavior': 'rocks', 'shade_preference': 'Low', 'category': 'material', 'commonness': 4, 'value': 45}
]

## 3b. **Spawn Logic and Effects**

### TODO: Determining resource spawning: location, success, types, quality

### Base probabilities of finding each resource

* There are 16 resources, each with a base "commonness" reflecting the probability that it should be spawned.
* These commonness values are the weights in a random choices function.
* These weights are modified by "mods" determined by variables at each level of the map hierarchy.
* The mods/probabilities could be passed down, or just the variables/state.

In [108]:
# base probability starts at the "commonness" of the
BASE_WEIGHTS = copy.deepcopy(RESOURCE_TYPES)
for i, resource in enumerate(RESOURCE_TYPES):
    BASE_WEIGHTS[i]["weight"] = resource["commonness"]

### world-level effects (time_of_day and rain)



**Overview**:
1. Different resource types spawn at different times of day, depending on their `spawn_behavior` (if the resources behave like bugs, plants, rocks or mushrooms). Currently:
  - Bugs more likely in AM Early.
  - Rocks more likely in AM Late.
  - Plants more likely in PM Early.
  - Mushrooms more likely in PM Late.

2. Different resource types are more likely to spawn in the rain, depending on their `spawn_behavior`. Currently:
  - Bugs and mushrooms more likely in rain.
  - Plants and rocks more likely in sun.


#### Time of day effects


In [109]:
#
# World-level effects
#

# Defines functions to apply to modify resource spawn probabilities
mods = {}
mods["world"] = []

# Defines multipliers on the probabilities of spawning a particular resource
effects = {}
effects["time_of_day"] = {}
effects["time_of_day"]["spawn_behavior"] = {
    'AM Early': {
        'bugs': 2
    },
    'AM Late': {
        'rocks': 2,
        'bugs': 1
    },
    'PM Early': {
        'plants': 2,
        'bugs': 1
    },
    'PM Late': {
        'mushrooms': 2,
    }
}

# Time of day effects
def time_of_day_effects(base_p, vars):
    p = copy.deepcopy(base_p)
    indep_var = "time_of_day"
    prop = "spawn_behavior"

    current_val = str(vars[indep_var])
    current_multipliers = effects.get(indep_var, {}).get(prop, {}).get(current_val, {})

    for resource in p:
        for key, value in current_multipliers.items():
            if resource[prop] == key:
                resource["weight"] = resource["weight"] * value
    return p

mods["world"].append(time_of_day_effects)

#### Rain condition affects resource type via spawn_behavior

In [110]:
effects["rain_condition"] = {}
effects["rain_condition"]["spawn_behavior"] = {
    'Rainy': {
        'bugs': 2,
        'mushrooms': 2,
    },
    'Sunny': {
        'plants': 2,
        'rocks': 2
    }
}

# Weather effects
def weather_effects(base_p, vars):
    p = copy.deepcopy(base_p)
    indep_var = "rain_condition"
    prop = "spawn_behavior"

    current_val = str(vars[indep_var])
    current_multipliers = effects.get(indep_var, {}).get(prop, {}).get(current_val, {})

    for resource in p:
        for key, value in current_multipliers.items():
            if resource[prop] == key:
                resource["weight"] = resource["weight"] * value
    return p

mods["world"].append(weather_effects)

### island effects (base probabilities, faction rep)

* Each faction's resources can spawn on any island.
* The probabilities of spawning a given faction's resources depend on island.

In [111]:
ISLAND_BASE_PROBS = {
    'Growth': {'Growth': 6, 'Shadow': 2, 'Light': 3, 'Stability': 1},
    'Stability': {'Growth': 1, 'Shadow': 3, 'Light': 2, 'Stability': 6},
    'Light': {'Growth': 3, 'Shadow': 1, 'Light': 6, 'Stability': 2},
    'Shadow': {'Growth': 2, 'Shadow': 6, 'Light': 1, 'Stability': 3},
}

effects["island_name"] = {}
effects["island_name"]["faction"] = ISLAND_BASE_PROBS

def island_effects(base_p, vars):
    p = copy.deepcopy(base_p)
    indep_var = "island_name"
    prop = "faction"

    current_val = str(vars[indep_var])
    current_multipliers = effects.get(indep_var, {}).get(prop, {}).get(current_val, {})

    for resource in p:
        for key, value in current_multipliers.items():
            if resource[prop] == key:
                resource["weight"] = resource["weight"] * value

    return p

def faction_rep_effects(base_p, vars):
    p = base_p.copy()
    return p;

mods["island"] = [island_effects]

### plot-level effects

In [112]:
#
# Plot-level effects
#

# independent variables
effects["shade_level"] = {}
effects["light_color"] = {}

# dependent variables
effects["shade_level"]["shade_preference"] = {
    'High': {
        'High': 4,
        'Medium': 2
    },
    'Medium': {
        'Medium': 2,
    },
    'Low': {
        'Low': 4,
        'Medium': 2
    }
}

# dependent variables
effects["light_color"]["category"] = {
    'Green': {
        'fruit': 4
    },
    'Yellow': {
        'sweet': 4,
    },
    'Purple': {
        'magic': 4
    },
    'Blue': {
        'material': 4,
    }
}

# Shade effects
def shade_effects(base_p, vars):
    p = copy.deepcopy(base_p)

    indep_var = "shade_level"
    prop = "spawn_behavior"

    current_val = str(vars[indep_var])
    current_multipliers = effects.get(indep_var, {}).get(prop, {}).get(current_val, {})

    for resource in p:
        for key, value in current_multipliers.items():
            if resource[prop] == key:
                resource["weight"] = resource["weight"] * value

    return p

# Light effects (assuming light_color is a value from 0 to 100, where 100 is bluest)
def light_effects(base_p, vars):
    p = copy.deepcopy(base_p)

    indep_var = "light_color"
    prop = "spawn_behavior"

    current_val = str(vars[indep_var])
    current_multipliers = effects.get(indep_var, {}).get(prop, {}).get(current_val, {})

    for resource in p:
        for key, value in current_multipliers.items():
            if resource[prop] == key:
                resource["weight"] = resource["weight"] * value

    return p

mods["plot"] = [shade_effects, light_effects]

### patch-level effects

In [113]:
#
# Patch-level effects
#

# Water effects
def water_effects(base_p, vars):
    p = copy.deepcopy(base_p)

    indep_var = "water_level"
    prop = "spawn_behavior"

    # map (1-100) to categorical (low/med/high)
    current_val = str(vars[indep_var])
    current_multipliers = effects.get(indep_var, {}).get(prop, {}).get(current_val, {})

    for resource in p:
        for key, value in current_multipliers.items():
            if resource[prop] == key:
                resource["weight"] = resource["weight"] * value
    return p

# Soil effects
def soil_effects(soil_quality, base_p):
    adjustments = copy.deepcopy(base_p)

    if 0 <= soil_quality <= 33:     # Low-quality soil
        adjustments['rocks'] *= 1     # More rocks found in low-quality soil
        adjustments['plants'] *= 1    # Plants don't thrive in low-quality soil
    elif 34 <= soil_quality <= 66:  # Medium-quality soil
        adjustments['plants'] *= 1    # No effect of medium-quality soil
    else:  # High-quality soil
        adjustments['plants'] *= 1    # Plants thrive in high-quality soil
        adjustments['bugs'] *= 1      # Bugs are attracted to rich soil
        adjustments['mushrooms'] *= 1  # Mushrooms grow better in rich soil

    return adjustments;

# mods["patch"] = [water_effects, soil_effects]

### node effects (x, y)

In [114]:
mods["node"] = []

# 4. Recipes and Crafting

## Crafting Overview
Players can craft four categories of items:

1. Foods
2. Health Potions
3. Growth Potions
4. Gifts

Each category of items has: 1 of 4 themes, and 1 of 4 item types. Each category has 16 unique recipes. These items are crafted by combining two input resources, via the logic described in the following section.

Items have:
* a category (food, gift, health_potion, growth_potion)
* a theme (e.g. "Sports", "Cookies")
* a type (e.g. "Baseball Glove", "Snickerdoodle")
* a faction (Light, Shadow, Stability, Growth)
* a faction_blend (Specific, Balanced, Synergistic, Neutral)

**Open Questions**:
1. Are which items that can be crafted be constrained by context (e.g. in the Medical Building, the player can only craft Potions)?

## Item Categories, Themes and Types

In [115]:
GIFT_THEMES = ACTIVITY_THEMES
FOOD_THEMES = ["Cupcake", "Ice Cream", "Cookie", "Candy"]
HEALTH_POTION_THEMES = ["Tea", "Soda", "Juice", "Coffee"]
GROWTH_POTION_THEMES = ["Theme1", "Theme2", "Theme3", "Theme4"]

# Change to gift types per activity
GIFT_TYPES_BY_THEME = {
    "Sports": ["Football", "Soccer Ball", "Baseball Glove", "Tennis Racket"],
    "Art": ["Paint Brush", "Sketch Pad", "Quill and Ink", "Clay"],
    "Music": ["Tap Shoes", "Guitar", "Drums", "Piano"],
    "Reading": ["Pop-Up Book", "Coloring Book", "Story Book", "Comic Book"]
}

# I gave the creature a X
GIFTS_BY_THEME_BY_ACTIVITY_TYPE = {
    "Sports": {
        "Playing Football": "Football",
        "Playing Soccer": "Soccer Ball",
        "Playing Baseball": "Baseball Glove",
        "Playing Tennis": "Tennis Racket"
    },
    "Art": {
        "Painting": "Paint Brush",
        "Sketching": "Sketch Pad",
        "Writing": "Quill and Ink",
        "Sculpting": "Clay"
    },
    "Music": {
        "Tap Dancing": "Pair of Tap Shoes",
        "Playing Guitar": "Guitar",
        "Playing Drums": "Snare Drum",
        "Playing Piano": "Piano"
    },
    "Reading": {
        "Reading Fiction": "Story Book",
        "Reading Comic Books": "Comic Book",
        "Reading Magazines": "Magazine",
        "Reading Poetry": "Book of Poems"
    }
}

FOOD_TYPES_BY_THEME = {
    "Cupcake": ["Ginger Cupcake", "Vanilla Cupcake", "Chocolate Cupcake", "Strawberry Cupcake"],
    "Ice Cream": ["Vanilla Ice Cream", "Chocolate Ice Cream", "Pistachio Ice Cream", "Rocky Road Ice Cream"],
    "Cookie": ["Chocolate Chip Cookie", "Snickerdoodle Cookie", "Oatmeal Cookie", "Macaroon Cookie"],
    "Candy": ["Sour Candy", "Sweet Candy", "Spicy Candy", "Crunchy Candy"]
}

HEALTH_POTION_TYPES_BY_THEME = {
    "Tea": ["Green Tea", "Milk Tea", "Oolong Tea", "Jasmine Tea"],
    "Soda": ["Cola", "Orange Soda", "Root Beer", "Lemon Lime Soda"],
    "Juice": ["Apple Juice", "Grapefruit Juice", "Guava Juice", "Tangerine Juice"],
    "Coffee": ["Espresso", "Mocha", "Latte", "Frappe"]
}

GROWTH_POTION_TYPES_BY_THEME = {
    "Theme1": ["Theme1A", "Theme1B", "Theme1C", "Theme1D"],
    "Theme2": ["Theme2A", "Theme2B", "Theme2C", "Theme2D"],
    "Theme3": ["Theme3A", "Theme3B", "Theme3C", "Theme3D"],
    "Theme4": ["Theme4A", "Theme4B", "Theme4C", "Theme4D"]
}

ITEM_THEMES_BY_CATEGORY = {
    "food": FOOD_THEMES,
    "gift": GIFT_THEMES,
    "health_potion": HEALTH_POTION_THEMES,
    "growth_potion": GROWTH_POTION_THEMES
}

ITEM_TYPES_BY_CATEGORY_BY_THEME = {
    "food": FOOD_TYPES_BY_THEME,
    "gift": GIFT_TYPES_BY_THEME,
    "health_potion": HEALTH_POTION_TYPES_BY_THEME,
    "growth_potion": GROWTH_POTION_TYPES_BY_THEME
}

## Crafting Logic

### Item Category determined by Resource Categories

Every recipe combines exactly two resources. As currently specified, the ordering of the two resources does not matter. However, these resources can be thought of as a primary, "base" resource, and a secondary, "modifier" resource.

* The combination of base and modifier resource categories (fruit, sweet, magic, material) determine the category of item (food, health potion, growth potion, or gift). Ex: combining one of 4 Fruit resources with one of 4 Sweet resources crafts one of the 16 Food items.

* After the item category is determined, to select which of the 16 items are crafted:
* The faction of the base resource determines the item/recipe faction.
* The faction of the modifier resource determines the item theme.

This combination logic is illustrated in the recipe spreadsheet screenshots.

---

#### **Resource Category Combinations: Base and Modifier Ingredients**

- **Foods**: Fruit (base) + Sweet (modifier)
  - **Example**: Dawnspore (Light, Fruit) as the base + Lumina Dew (Light, Sweet) as the modifier = Light-specific Food Item

- **Health Potions**: Fruit (base) + Magic (modifier)
  - **Example**: Shadowbug Nectar (Shadow, Sweet) as the base + Beetle Wing (Shadow, Magic) as the modifier = Shadow-specific Health Potion

- **Growth Potions**: Sweet (base) + Magic (modifier)
  - **Example**: Lifeseed (Growth, Food) as the base + Tranquil Moss (Growth, Magic) as the modifier = Growth-specific Growth Potion

- **Gifts**: Material (base) + Magic (modifier)
  - **Example**: Bedrock (Stability, Material) as the base + Runestone Shards (Stability, Magic) as the modifier = Stability-specific Gift

In [116]:
# defines the category for any order of resources
RECIPE_CATEGORIES = {
    frozenset(["fruit", "sweet"]): "food",
    frozenset(["fruit", "magic"]): "health_potion",
    frozenset(["sweet", "magic"]): "growth_potion",
    frozenset(["material", "magic"]): "gift"
}

# defines the base and mod ingredient for any order of resources
RECIPE_BASES_AND_MODS = {
    frozenset(["fruit", "sweet"]): {"base": "fruit", "mod": "sweet"},
    frozenset(["fruit", "magic"]): {"base": "fruit", "mod": "magic"},
    frozenset(["sweet", "magic"]): {"base": "sweet", "mod": "magic"},
    frozenset(["material", "magic"]): {"base": "material", "mod": "magic"}
}

# returns item category (gift, food, health_potion, growth_potion) from two ingredient resources
def get_item_category(resource1, resource2):
    resource_categories = frozenset([resource1["category"], resource2["category"]])

    try:
        item_category = RECIPE_CATEGORIES[resource_categories]
    except KeyError:
        return None
        # raise ValueError(f"Invalid combination of categories: {resource_categories}")

    return item_category

### Item Theme and Type determined by Modifier and Base Resource Factions

Similar to that faction having recipes, I just didn't store the recipes in that way above so indexed by theme first.


In [117]:
# finds base, mod resource factions
def get_base_and_mod_factions(resource1, resource2):
    resource_categories = frozenset([resource1["category"], resource2["category"]])

    try:
        base_category = RECIPE_BASES_AND_MODS[resource_categories]["base"]
    except KeyError:
        return None
        # raise ValueError(f"Invalid combination of categories: {resource_categories}")

    if resource1["category"] == base_category:
        base_faction = resource1["faction"]
        mod_faction = resource2["faction"]
    else:
        base_faction = resource2["faction"]
        mod_faction = resource1["faction"]

    return base_faction, mod_faction

# determine item type from resources
def get_item_theme_and_type(resource1, resource2):
    resource_categories = frozenset([resource1["category"], resource2["category"]])

    # get item category (food, gift, health_potion, growth_potion)
    item_category = get_item_category(resource1, resource2)

    # get base_faction and mod_faction
    base_faction, mod_faction = get_base_and_mod_factions(resource1, resource2)

    # theme is index of mod_faction in THEMES array
    item_theme = ITEM_THEMES_BY_CATEGORY[item_category][FACTIONS.index(mod_faction)]
    themed_recipes = ITEM_TYPES_BY_CATEGORY_BY_THEME[item_category][item_theme]
    item_type = themed_recipes[FACTIONS.index(base_faction)]

    return item_theme, item_type

### Faction Blend determined by Modifier and Base Resource Factions

Crafted items will inherit a "faction blend," determined by the factions of the base and modifier resources. The algorithmic logic for determining this blend is as follows:

1. **Specific**: If both resources are from the same faction, the item blend is "Faction Special".

2. **Synergistic**: If the factions of the resources are known to work well together (Light with Growth, or Shadow with Stability), the item blend is "Synergistic".

3. **Balanced**: If the factions of the resources are natural opposites (Light with Shadow, or Growth with Stability), the item blend is "Balanced".

4. **Neutral**: If the factions of the resources don't fall into any of the above categories (Light with Stability, or Shadow with Growth), the item blend is "Mixed".

In [118]:
def get_item_blend(resource1, resource2):
    # Determine Potion Type
    item_blend = None

    type1 = resource1["faction"]
    type2 = resource2["faction"]

    if (type1, type2) in BALANCED_FACTION_COMBINATIONS or (type2, type1) in BALANCED_FACTION_COMBINATIONS:
        item_blend = "Balanced"
    elif (type1, type2) in SYNERGISTIC_FACTION_COMBINATIONS or (type2, type1) in SYNERGISTIC_FACTION_COMBINATIONS:
        item_blend = "Synergistic"
    elif (type1, type2) in NEUTRAL_FACTION_COMBINATIONS or (type2, type1) in NEUTRAL_FACTION_COMBINATIONS:
        item_blend = "Neutral"
    elif (type1 == type2):
        item_blend = "Special"

    return item_blend

# Example usage (order doesn't matter):
print(get_item_theme_and_type(RESOURCE_TYPES[0], RESOURCE_TYPES[1]))  # Output: "Crunchy Candy"
print(get_item_theme_and_type(RESOURCE_TYPES[5], RESOURCE_TYPES[8]))  # Output: "Chocolate Chip Cookie"
print(get_item_theme_and_type(RESOURCE_TYPES[5], RESOURCE_TYPES[12])) # Output: "Snickerdoodle Cookie"

# Example usage (order doesn't matter):
print(get_item_blend(RESOURCE_TYPES[0], RESOURCE_TYPES[1]))  # Output: "Special"
print(get_item_blend(RESOURCE_TYPES[1], RESOURCE_TYPES[0]))  # Output: "Special"
print(get_item_blend(RESOURCE_TYPES[0], RESOURCE_TYPES[5]))  # Output: "Balanced"
print(get_item_blend(RESOURCE_TYPES[0], RESOURCE_TYPES[10])) # Output: "Synergistic"

('Candy', 'Crunchy Candy')
('Cookie', 'Chocolate Chip Cookie')
('Cookie', 'Snickerdoodle Cookie')
Special
Special
Balanced
Synergistic


## CraftingTable

In [119]:
class CraftingTable:
    def __init__(self):
        self.trash = []

    def get_trash(self):
        return self.trash

    def craft(self, resource1, resource2):
        item_category = get_item_category(resource1, resource2)
        if item_category:
            item_theme, item_type = get_item_theme_and_type(resource1, resource2)
            item_quality = (resource1["resource_quality"] + resource2["resource_quality"])
            base_faction, mod_faction = get_base_and_mod_factions(resource1, resource2)
            item_faction = base_faction
            item_blend = get_item_blend(resource1, resource2)

            item = { "item_category": item_category, "item_theme": item_theme, "item_faction": item_faction, "item_blend": item_blend, "item_type": item_type, "item_quality": item_quality }
            self.trash.append([resource1, resource2])
        else:
            item = None

        return item

# 5. Item Effects

## Base Effects by Item Category (gift, potion, food)

**Overview**:
Each item category has an effect on creature stats.
- **Health potions** have most effect on creature health, and much smaller effects on creature mood or social stats.
- **Gifts** have no effect on creature health, but impact creature mood and social (with a slightly larger effect on social).
- **Foods** impact all stats roughly equally, though have the largest effect on mood.
- **Growth potions** do nothing to creatures, and can not be gifted or applied to them.

In [120]:
ITEM_BASE_EFFECTS = {
    "health_potion": {
        "health": 13,
        "mood": 3,
        "social": 2
    },
    "food": {
        "health": 5,
        "mood": 7,
        "social": 6
    },
    "gift": {
        "health": 0,
        "mood": 8,
        "social": 10
    },
    "growth_potion": {
        "health": 0,
        "mood": 0,
        "social": 0
    }
}

CREATURE_CHOICES["item_effects"] = ITEM_BASE_EFFECTS

## Mods for Faction, pastime on item effects

In [121]:
#
# Faction-level effects
#

# Factions have item theme preferences (matching their activity preferences)

mods_to_base_item_effects = {}
mods_to_base_item_effects["faction_name"] = {}

b = {
    'rest': { # HP Boost
        'health_potion': { "hangout": 1, "rest": 0, "activity": 0, "tend": 0 },
        'growth_potion': { "hangout": 0, "rest": 1, "activity": 0, "tend": 0 },
        'gift': { "hangout": 0, "rest": 0, "activity": 1, "tend": 0 },
        'food': { "hangout": 0, "rest": 0, "activity": 0, "tend": 1 }
    },
    'hangout': { # Food Boost
        'health_potion': { "hangout": 1, "rest": 0, "activity": 0, "tend": 0 },
        'growth_potion': { "hangout": 0, "rest": 1, "activity": 0, "tend": 0 },
        'gift': { "hangout": 0, "rest": 0, "activity": 1, "tend": 0 },
        'food': { "hangout": 0, "rest": 0, "activity": 0, "tend": 1 }
    },
    'activity': { # Gift Boost
        'health_potion': { "hangout": 1, "rest": 0, "activity": 0, "tend": 0 },
        'growth_potion': { "hangout": 0, "rest": 1, "activity": 0, "tend": 0 },
        'gift': { "hangout": 0, "rest": 0, "activity": 1, "tend": 0 },
        'food': { "hangout": 0, "rest": 0, "activity": 0, "tend": 1 }
    },
    'tend': { # GP Boost
        'health_potion': { "hangout": 1, "rest": 0, "activity": 0, "tend": 0 },
        'growth_potion': { "hangout": 0, "rest": 1, "activity": 0, "tend": 0 },
        'gift': { "hangout": 0, "rest": 0, "activity": 1, "tend": 0 },
        'food': { "hangout": 0, "rest": 0, "activity": 0, "tend": 1 }
    }
}

# Pastime choice depends on time_of_day and faction
def faction_time_of_day_pastime_effects(base_p, vars):
    p = copy.deepcopy(base_p)
    effects = p["item_effects"]

    indep_var1 = "faction_name"
    prop = "item_effects"

    current_val1 = str(vars.get(indep_var1, ""))

    if current_val1:
        effects_to_apply = mods_to_base_item_effects.get(indep_var1, {}).get(prop, {})
        current_multipliers = effects_to_apply.get(current_val1, {}).get(current_val2, {})
        for pastime in pastimes:
            for key, multiplier in current_multipliers.items():
                if pastime[prop] == key:
                    pastime["weight"] = pastime["weight"] * multiplier
    return p

# 6. Players

In [122]:
PLAYERS = [
    { "name": "Aisha", "island_preference": "Growth", "faction_preference": "Growth", "item_preference": "gift" },
    { "name": "Bianca", "island_preference": "Stability", "faction_preference": "Stability", "item_preference": "health_potion" },
    { "name": "Chiara", "island_preference": "Shadow", "faction_preference": "Shadow", "item_preference": "food" },
    { "name": "Dalia", "island_preference": "Light", "faction_preference": "Light", "item_preference": "gift" },
    { "name": "Emiko", "island_preference": "Growth", "faction_preference": "Growth", "item_preference": "health_potion" },
    { "name": "Fatima", "island_preference": "Stability", "faction_preference": "Stability", "item_preference": "food" },
    { "name": "Gisele", "island_preference": "Shadow", "faction_preference": "Shadow", "item_preference": "gift" },
    { "name": "Hannah", "island_preference": "Light", "faction_preference": "Light", "item_preference": "health_potion" },
    { "name": "Isabella", "island_preference": "Growth", "faction_preference": "Growth", "item_preference": "food" },
    { "name": "Jasmine", "island_preference": "Stability", "faction_preference": "Stability", "item_preference": "gift" },
    { "name": "Kira", "island_preference": "Shadow", "faction_preference": "Shadow", "item_preference": "health_potion" },
    { "name": "Lina", "island_preference": "Light", "faction_preference": "Light", "item_preference": "food" }
]

class Player:
    def __init__(self, player_data):
        self.name = player_data.get("name", "PlayerName")

        self.island_preference = player_data.get("island_preference", random.choice(ISLANDS)["name"])
        self.faction_preference = player_data.get("faction_preference", random.choice(FACTIONS))
        self.item_preference = player_data.get("item_preference", random.choice(["gift", "health_potion", "food"]))
        self.creature_style_preference = player_data.get("creature_style_preference", random.choice(CREATURE_STYLES))
        self.creature_color_preference = player_data.get("creature_color_preference", random.choice(CREATURE_COLORS))

        self.resource_inventory = []
        self.item_inventory = []

    def get_name(self):
        return self.name

    def get_preferences(self):
        return {
            "island_preference": self.island_preference,
            "faction_preference": self.faction_preference,
            "item_preference": self.item_preference,
            "creature_style_preference": self.creature_style_preference,
            "creature_color_preference": self.creature_color_preference
        }

    def migrate(self, world):
        # pick an island, island preference, then random plot
        island = random.choice([random.choice(world.children), world.get_island_by_name(self.island_preference)])
        plot = random.choice(island.children)

        return plot

    def pick_creature(self, creatures):
        # Preference based selection
        faction_preferred_creatures = [c for c in creatures if c.faction == self.faction_preference]
        style_preferred_creatures = [c for c in creatures if c.style == self.creature_style_preference]
        color_preferred_creatures = [c for c in creatures if c.color == self.creature_color_preference]

        preferred_creatures = faction_preferred_creatures or style_preferred_creatures or color_preferred_creatures or creatures

        return random.choice(preferred_creatures)

    def give_gift_to_creature(self, world, creature, giving_data):
        if self.item_inventory:
            item = random.choice(self.item_inventory)
            effects = creature.give_item(self, item)
            self.item_inventory.remove(item)

            giving_data.append({
                "day": world.clock.get_day(),
                "time_of_day": world.clock.get_time_of_day(),
                "day_of_week": world.clock.get_day_of_week(),
                "player": self.name,
                "creature": creature.get_name(),
                "item_category": item["item_category"],
                "item_type": item["item_type"],
                "item_faction": item["item_faction"],
                "item_quality": item["item_quality"],
                "health_effect": effects["health"],
                "mood_effect": effects["mood"],
                "social_effect": effects["social"]
            })

    def attempt_to_craft(self, world, crafting_table, crafting_data):
        # Attempt to craft an item from two resources
        for resource1 in self.resource_inventory:
            for resource2 in self.resource_inventory:
                if resource1 != resource2:
                    item = crafting_table.craft(resource1, resource2)
                    if item:
                        # Add the crafted item to the item_inventory
                        self.item_inventory.append(item)

                        # Remove the used resources from inventory if they exist
                        if resource1 in self.resource_inventory:
                            self.resource_inventory.remove(resource1)
                        if resource2 in self.resource_inventory:
                            self.resource_inventory.remove(resource2)

                        # Log the event
                        crafting_data.append({
                            "day": world.clock.get_day(),
                            "time_of_day": world.clock.get_time_of_day(),
                            "day_of_week": world.clock.get_day_of_week(),
                            "player": self.name,
                            "item_category": item["item_category"],
                            "item_faction": item["item_faction"],
                            "item_type": item["item_type"],
                            "item_quality": item["item_quality"]
                        })

    def next(self, world, creature_population, crafting_table, player_data):
        # go to a new location
        plot = self.migrate(world)

        # pick a nearby creature
        nearby_creatures = creature_population.get_creature_locations().get(plot.get_id(), [])
        if len(nearby_creatures) > 0:
            creature = player.pick_creature(nearby_creatures)
            if len(self.item_inventory) > 0:
                player.give_gift_to_creature(world, creature, player_data["giving"])

        # forage a few resources from that plot
        for i in range(random.randint(0,4)):
            node = plot.get_random_child().get_random_child()
            resource = node.spawn_resource()
            self.resource_inventory.append(resource)

            player_data["foraging"].append({
                "player": player.get_name(),
                "time_of_day": world.clock.get_time_of_day(),
                "is_raining": world.weather.get_rain_condition(),
                "island": plot.get_parent_island().get_name(),
                "plot_id": plot.get_id(),
                "resource_name": resource["name"],
                "resource_faction": resource["faction"],
                "resource_spawn_behavior": resource["spawn_behavior"],
                "resource_quality": resource["resource_quality"],
                "node_x": node.coords[0],
                "node_y": node.coords[1]
            })

        # attempt to craft anything new
        if len(self.resource_inventory) > 1:
            self.attempt_to_craft(world, crafting_table, player_data["crafting"])

# Map Access, Nighttime Updates, Newspaper, Shops/Requests

## Newspaper
- e.g. "Cookie Party" announcement
- Faction imbalance alerts
- Recipes/hints

# Next Steps

## Weather Patterns

### **1. Base Weather By Season (Default Patterns):**
Each island will have a default weather pattern for each season. This is what the island will generally experience in the absence of any external influences.

**Example Table** (you can fill this in with desired patterns):

| Island     | Spring  | Summer | Fall   | Winter |
|------------|---------|--------|--------|--------|
| **Growth** | Sunny/Warm | Rainy/Warm | Rainy/Cool | Sunny/Cool |
| **Shadow** | Rainy/Cool | Sunny/Cool | Rainy/Cool | Sunny/Cool |
| **Light**  | Sunny/Warm | Rainy/Warm | Sunny/Cool | Rainy/Cool |
| **Stability**| Rainy/Cool | Sunny/Warm | Rainy/Cool | Sunny/Warm |

### **2. Breeze Influence:**

The breeze acts as a mechanism to shift or rotate the weather patterns across the islands. The key here is that the breeze doesn't alter the inherent base weather of an island but temporarily borrows the weather from another island.

- **Northward Breeze**: Weather conditions from the bottom islands (Light & Stability) shift up to the top islands (Growth & Shadow).
  
- **Eastward Breeze**: Weather from the left islands (Growth & Light) shifts to the right islands (Shadow & Stability).
  
- **Southward Breeze**: The opposite of Northward, the weather conditions from the top islands (Growth & Shadow) shift down to the bottom islands (Light & Stability).
  
- **Westward Breeze**: The opposite of Eastward, weather from the right islands (Shadow & Stability) shifts to the left islands (Growth & Light).

### **3. No Breeze - Base Probabilities:**
In the absence of any breeze, the islands revert or continue with their default weather pattern for that season.

### **Framework Logic:**

1. **Start of Day**:
   - Check if there's a breeze for the day.
   - If no breeze, implement base weather probability for that island and season.
   - If there's a breeze, check the direction and shift the weather from the relevant island.

2. **End of Day**:
   - Register the weather for each island for that day.
   - Check if conditions for the next day will involve a breeze.
   - Prepare the weather patterns for the next day based on the above checks.

### **Strategic Implementation**:

1. **Forecasting**:
   - Provide players with occasional clues or forecasts about potential breezes, allowing them to predict changes in weather. This adds a layer of strategy and planning.

2. **Breeze Probabilities**:
   - Each season might have its own inherent likelihood of certain breezes occurring, adding depth to the mechanics. For example, Summer might have a higher likelihood of Eastward breezes.

By combining default weather patterns with the dynamic influence of breezes, players are presented with a challenge of both understanding the inherent nature of each island and adapting to the unpredictable changes the breeze brings. This mix of predictability and surprise ensures player engagement and strategic thinking.

## 6. Anomalies, Shops and Bridges

BADGES: "Treat 5 Creatures to 100 Health"

MURAL: Paint-by-badge

PAINTING LOG:
Player | Time | Mural Part
Jen | Day 1 | "Treat 5 Creatures"

PROBLEM SOLVED:
- DELIVER PROGRESSIONS SYSTEM TO PLAYER

ADDED CONSIDERATIONS:
- PUBLIC EXPERTISE
- COLLECTIVE COMPLETION

# Simulate

In [123]:
players = [Player(player_data) for player_data in PLAYERS]

start_day = 0
clock = WorldClock(start_day)
weather = Weather()

crafting_table = CraftingTable()

world = World(clock, weather, ISLANDS, mods, BASE_WEIGHTS) # BASE_WEIGHTS is a list of resources, with "weight" added
creature_population = CreaturePopulation(world, CREATURES, creature_mods, CREATURE_CHOICES)

In [141]:
import pandas as pd

# Data
player_data = {
    "foraging": [],
    "crafting": [],
    "giving": []
}
creature_data = {
    "stats": [],
    "migrations": []
}
world_data = []

# 56 = (14 days x 4 times of day) = One season
for i in range(112):
    world_data.append(world.get_level_variables())

    # creatures make moves
    creature_population.next(world, creature_data)

    # players make moves
    for player in players:
        player.next(world, creature_population, crafting_table, player_data)

    # tick forward in time
    world.next()

#world_df = pd.DataFrame(world_data)
#creature_stats = pd.DataFrame(creature_data["stats"])
#migration_df = pd.DataFrame(creature_data["migrations"])
#foraging_df = pd.DataFrame(player_data["foraging"])
#crafting_df = pd.DataFrame(player_data["crafting"])
#giving_df = pd.DataFrame(player_data["giving"])

# Data

## Foraging

In [142]:
foraging_df = pd.DataFrame(player_data["foraging"])
foraging_df

# Island preference
# Faction preference

# Where to find a resource?
# Where to find a resource type?
# Where to find a high quality resource?

,player,time_of_day,is_raining,island,plot_id,resource_name,resource_faction,resource_spawn_behavior,resource_quality,node_x,node_y
0,Aisha,AM Early,Rainy,Growth,Growth > Large Clearing,Tranquil Moss,Growth,plants,100,20,0
1,Bianca,AM Early,Rainy,Stability,Stability > Small Clearing,Beetle Wing,Shadow,bugs,60,-37,-24
2,Bianca,AM Early,Rainy,Stability,Stability > Small Clearing,Fungus Fiber,Light,mushrooms,81,-34,-25
3,Bianca,AM Early,Rainy,Stability,Stability > Small Clearing,Dark Silk,Shadow,bugs,59,-39,-25
4,Chiara,AM Early,Rainy,Stability,Stability > South Fields,Duskworm Fruit,Shadow,bugs,63,-11,-22
...,...,...,...,...,...,...,...,...,...,...,...
2661,Jasmine,PM Late,Sunny,Light,Light > Secret Forest,Lumina Dew,Light,mushrooms,93,13,-10
2662,Kira,PM Late,Sunny,Shadow,Shadow > South Fields,Stonefruit,Stability,rocks,53,-12,20
2663,Kira,PM Late,Sunny,Shadow,Shadow > South Fields,Bedrock,Stability,rocks,63,-12,22
2664,Kira,PM Late,Sunny,Shadow,Shadow > South Fields,Beetle Wing,Shadow,bugs,96,-13,21


## Crafting

In [143]:
crafting_df = pd.DataFrame(player_data["crafting"])
crafting_df

# Looking at item recipes and/or crafting attempts?
# Item preference
# Who crafts the best items?

,day,time_of_day,day_of_week,player,item_category,item_faction,item_type,item_quality
0,6,AM Early,Sunday,Aisha,gift,Stability,Soccer Ball,171
1,6,AM Early,Sunday,Bianca,gift,Stability,Guitar,114
2,6,AM Early,Sunday,Chiara,health_potion,Shadow,Oolong Tea,113
3,6,AM Early,Sunday,Emiko,food,Growth,Ginger Cupcake,166
4,6,AM Early,Sunday,Hannah,gift,Stability,Guitar,187
...,...,...,...,...,...,...,...,...
1386,33,PM Late,Saturday,Isabella,health_potion,Stability,Orange Soda,148
1387,33,PM Late,Saturday,Isabella,gift,Stability,Soccer Ball,163
1388,33,PM Late,Saturday,Jasmine,food,Light,Strawberry Cupcake,124
1389,33,PM Late,Saturday,Kira,health_potion,Stability,Grapefruit Juice,117


In [144]:
import numpy as np
from google.colab import autoviz

def heatmap_avg(df, x_colname, y_colname, value_colname, figscale=1, mpl_palette_name='viridis'):
    from matplotlib import pyplot as plt
    import seaborn as sns
    import pandas as pd

    plt.subplots(figsize=(8 * figscale, 8 * figscale))
    pivot_table = df.pivot_table(values=value_colname,
                                 index=y_colname,
                                 columns=x_colname,
                                 aggfunc=np.mean)  # Using mean for average. Replace with np.sum for total.

    sns.heatmap(pivot_table, cmap=mpl_palette_name)
    plt.xlabel(x_colname)
    plt.ylabel(y_colname)
    return autoviz.MplChart.from_current_mpl_state()

chart = heatmap_avg(crafting_df, 'item_category', 'item_faction', 'item_quality')
chart


In [145]:
import numpy as np
from google.colab import autoviz

def scatter_plots(df, colname_pairs, figscale=1, alpha=.8):
  from matplotlib import pyplot as plt
  plt.figure(figsize=(len(colname_pairs) * 6 * figscale, 6 * figscale))
  for plot_i, (x_colname, y_colname) in enumerate(colname_pairs, start=1):
    ax = plt.subplot(1, len(colname_pairs), plot_i)
    df.plot(kind='scatter', x=x_colname, y=y_colname, s=(32 * figscale), alpha=alpha, ax=ax)
    ax.spines[['top', 'right',]].set_visible(False)
  plt.tight_layout()
  return autoviz.MplChart.from_current_mpl_state()

chart = scatter_plots(crafting_df, *[[['item_faction', 'item_quality']]], **{})
chart

In [146]:
import numpy as np
from google.colab import autoviz

def scatter_plots(df, colname_pairs, figscale=1, alpha=.8):
  from matplotlib import pyplot as plt
  plt.figure(figsize=(len(colname_pairs) * 6 * figscale, 6 * figscale))
  for plot_i, (x_colname, y_colname) in enumerate(colname_pairs, start=1):
    ax = plt.subplot(1, len(colname_pairs), plot_i)
    df.plot(kind='scatter', x=x_colname, y=y_colname, s=(32 * figscale), alpha=alpha, ax=ax)
    ax.spines[['top', 'right',]].set_visible(False)
  plt.tight_layout()
  return autoviz.MplChart.from_current_mpl_state()

chart = scatter_plots(crafting_df, *[[['item_category', 'item_quality']]], **{})
chart

## Creature Migrations

In [147]:
migration_df = pd.DataFrame(creature_data["migrations"])

tending_df = migration_df[migration_df["pastime"] == "tend"]
tending_df

,time,creature_name,creature_faction,creature_style,creature_color,pastime,plot_id,x,y,island,activity_theme,activity
1,0,Prismaros,Light,B,Blue,tend,Shadow > Secret Forest,12,30,Shadow,NaN,NaN
7,0,Iris,Light,C,Purple,tend,Light > Small Garden,12,20,Light,NaN,NaN
11,0,Blaze,Light,B,Red,tend,Shadow > Musical Mines,26,6,Shadow,NaN,NaN
12,0,Radiance,Light,C,Red,tend,Light > Small Garden,12,20,Light,NaN,NaN
14,0,Phoenix,Light,E,Red,tend,Light > Large Clearing,12,0,Light,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
11178,3,Shieldstone,Stability,D,Blue,tend,Shadow > Musical Mines,26,6,Shadow,NaN,NaN
11180,3,Ash,Stability,A,Purple,tend,Stability > South Fields,27,18,Stability,NaN,NaN
11185,3,Ambrose,Stability,A,Red,tend,Stability > North Fields,24,27,Stability,NaN,NaN
11194,3,Endurance,Stability,E,Green,tend,Light > Large Garden,0,24,Light,NaN,NaN


## Creature Activities

In [148]:
migration_df = pd.DataFrame(creature_data["migrations"])

activities_df = migration_df[migration_df['pastime'] == 'activity']
activities_df

,time,creature_name,creature_faction,creature_style,creature_color,pastime,plot_id,x,y,island,activity_theme,activity
5,0,Amethyst,Light,A,Purple,activity,Growth > North Fields,24,27,Growth,Sports,Playing Football
21,0,Lumos,Light,B,Orange,activity,Growth > Secret Forest,12,30,Growth,Reading,Reading Fiction
27,0,Duskray,Shadow,C,Blue,activity,Shadow > Community Garden,15,9,Shadow,Art,Sculpting
28,0,Twilight,Shadow,D,Blue,activity,Shadow > Musical Mines,26,6,Shadow,Music,Playing Piano
30,0,Obsidian,Shadow,A,Purple,activity,Stability > Musical Mines,26,6,Stability,Music,Playing Guitar
...,...,...,...,...,...,...,...,...,...,...,...,...
11170,3,Solstice,Growth,A,Orange,activity,Stability > North Fields,24,27,Stability,Sports,Playing Soccer
11172,3,Radiant Blossom,Growth,C,Orange,activity,Stability > Musical Mines,26,6,Stability,Music,Playing Guitar
11184,3,Solidarity,Stability,E,Purple,activity,Light > North Fields,24,27,Light,Sports,Playing Baseball
11188,3,Resolute,Stability,D,Red,activity,Stability > Secret Forest,12,30,Stability,Reading,Reading Comic Books


## Player Gifting

In [149]:
giving_df = pd.DataFrame(player_data["giving"])

## Creature Stats

In [150]:
creature_stats = pd.DataFrame(creature_data["stats"])

# Visualizations

## Activity theme by creature faction

In [132]:
faction_by_activity_theme

NameError: ignored

## Creature activity by faction

In [151]:
activity_by_faction

NameError: ignored

In [ ]:
faction_health_chart

## Creatures: Creature Faction vs Island

In [ ]:
chart

## Foraging: Resource Faction by Time of Day

In [ ]:
chart2

## Foraging: Resource Island by Time of Day

In [ ]:
chart3

## Visualizations Code

#### Faction by Activity Theme

In [154]:
import numpy as np
from google.colab import autoviz

def heatmap(df, x_colname, y_colname, figscale=1, mpl_palette_name='viridis'):
  from matplotlib import pyplot as plt
  import seaborn as sns
  import pandas as pd
  plt.subplots(figsize=(8 * figscale, 8 * figscale))
  df_2dhist = pd.DataFrame({
      x_label: grp[y_colname].value_counts()
      for x_label, grp in df.groupby(x_colname)
  })
  sns.heatmap(df_2dhist, cmap=mpl_palette_name)
  plt.xlabel(x_colname)
  plt.ylabel(y_colname)
  return autoviz.MplChart.from_current_mpl_state()

faction_by_activity_theme = heatmap(activities_df, *['creature_faction', 'activity_theme'], **{})
faction_by_activity_theme

In [155]:
import numpy as np
from google.colab import autoviz

def heatmap(df, x_colname, y_colname, figscale=1, mpl_palette_name='viridis'):
  from matplotlib import pyplot as plt
  import seaborn as sns
  import pandas as pd
  plt.subplots(figsize=(8 * figscale, 8 * figscale))
  df_2dhist = pd.DataFrame({
      x_label: grp[y_colname].value_counts()
      for x_label, grp in df.groupby(x_colname)
  })
  sns.heatmap(df_2dhist, cmap=mpl_palette_name)
  plt.xlabel(x_colname)
  plt.ylabel(y_colname)
  return autoviz.MplChart.from_current_mpl_state()

growth_style_activity = activities_df[activities_df["creature_faction"] == "Growth"]
light_style_activity = activities_df[activities_df["creature_faction"] == "Light"]
shadow_style_activity = activities_df[activities_df["creature_faction"] == "Shadow"]
stability_style_activity = activities_df[activities_df["creature_faction"] == "Stability"]

growth_style_by_activity = heatmap(growth_style_activity, *['creature_style', 'activity_theme'], **{})
light_style_by_activity = heatmap(light_style_activity, *['creature_style', 'activity_theme'], **{})
shadow_style_by_activity = heatmap(shadow_style_activity, *['creature_style', 'activity_theme'], **{})
stability_style_by_activity = heatmap(stability_style_activity, *['creature_style', 'activity_theme'], **{})

light_style_by_activity

#### Activity Type by Faction

In [ ]:
import numpy as np
from google.colab import autoviz

def heatmap(df, x_colname, y_colname, x_order=None, figscale=1, mpl_palette_name='viridis'):
    from matplotlib import pyplot as plt
    import seaborn as sns
    import pandas as pd
    plt.subplots(figsize=(8 * figscale, 8 * figscale))

    df_2dhist = pd.DataFrame({
        x_label: grp[y_colname].value_counts()
        for x_label, grp in df.groupby(x_colname)
    })

    # If an order for the x-axis is provided, use it
    if x_order:
        df_2dhist = df_2dhist[x_order]

    sns.heatmap(df_2dhist, cmap=mpl_palette_name)
    plt.xlabel(x_colname)
    plt.ylabel(y_colname)
    return autoviz.MplChart.from_current_mpl_state()

# Assume your array of names for ordering is:
names_order = ['Reading Poetry', 'Reading Magazines', 'Reading Fiction', 'Reading Comic Books', 'Painting', 'Sketching', 'Writing', 'Sculpting', 'Playing Guitar', 'Tap Dancing', 'Playing Piano', 'Playing Drums', 'Playing Football', 'Playing Baseball', 'Playing Soccer', 'Playing Tennis']

activity_by_faction = heatmap(activities_df, 'activity', 'creature_faction', x_order=names_order)

In [ ]:
# final day stats
creature_stats = pd.DataFrame(creature_data["stats"])
creature_stats

### Faction Activities

In [ ]:
import numpy as np
from google.colab import autoviz

def heatmap(df, x_colname, y_colname, figscale=1, mpl_palette_name='viridis'):
  from matplotlib import pyplot as plt
  import seaborn as sns
  import pandas as pd
  plt.subplots(figsize=(8 * figscale, 8 * figscale))
  df_2dhist = pd.DataFrame({
      x_label: grp[y_colname].value_counts()
      for x_label, grp in df.groupby(x_colname)
  })
  sns.heatmap(df_2dhist, cmap=mpl_palette_name)
  plt.xlabel(x_colname)
  plt.ylabel(y_colname)
  return autoviz.MplChart.from_current_mpl_state()

faction_activities = heatmap(activities_df, *['creature_faction', 'activity_theme'], **{})

### Creatures: Stats by Faction

In [ ]:
import numpy as np
from google.colab import autoviz

def scatter_plots(df, colname_pairs, figscale=1, alpha=.8):
  from matplotlib import pyplot as plt
  plt.figure(figsize=(len(colname_pairs) * 6 * figscale, 6 * figscale))
  for plot_i, (x_colname, y_colname) in enumerate(colname_pairs, start=1):
    ax = plt.subplot(1, len(colname_pairs), plot_i)
    df.plot(kind='scatter', x=x_colname, y=y_colname, s=(32 * figscale), alpha=alpha, ax=ax)
    ax.spines[['top', 'right',]].set_visible(False)
  plt.tight_layout()
  return autoviz.MplChart.from_current_mpl_state()

faction_health_chart = scatter_plots(creature_stats, *[[['creature_faction', 'creature_mood'], ['creature_faction', 'creature_health']]], **{})

### Creatures: Creature Faction vs Island

In [ ]:
import numpy as np
from google.colab import autoviz

def heatmap(df, x_colname, y_colname, figscale=1, mpl_palette_name='viridis'):
  from matplotlib import pyplot as plt
  import seaborn as sns
  import pandas as pd
  plt.subplots(figsize=(8 * figscale, 8 * figscale))
  df_2dhist = pd.DataFrame({
      x_label: grp[y_colname].value_counts()
      for x_label, grp in df.groupby(x_colname)
  })
  sns.heatmap(df_2dhist, cmap=mpl_palette_name)
  plt.xlabel(x_colname)
  plt.ylabel(y_colname)
  return autoviz.MplChart.from_current_mpl_state()

chart = heatmap(migration_df, *['creature_faction', 'island'], **{})

### Foraging: Resource Faction by Island

In [ ]:
import numpy as np
from google.colab import autoviz

def heatmap(df, x_colname, y_colname, figscale=1, mpl_palette_name='viridis'):
  from matplotlib import pyplot as plt
  import seaborn as sns
  import pandas as pd
  plt.subplots(figsize=(8 * figscale, 8 * figscale))
  df_2dhist = pd.DataFrame({
      x_label: grp[y_colname].value_counts()
      for x_label, grp in df.groupby(x_colname)
  })
  sns.heatmap(df_2dhist, cmap=mpl_palette_name)
  plt.xlabel(x_colname)
  plt.ylabel(y_colname)
  return autoviz.MplChart.from_current_mpl_state()

resource_faction_by_island = heatmap(foraging_df, *['island', 'resource_faction'], **{})

### Foraging: Resource Faction by Time of Day

In [ ]:
import numpy as np
from google.colab import autoviz

def heatmap(df, x_colname, y_colname, figscale=1, mpl_palette_name='viridis'):
  from matplotlib import pyplot as plt
  import seaborn as sns
  import pandas as pd
  plt.subplots(figsize=(8 * figscale, 8 * figscale))
  df_2dhist = pd.DataFrame({
      x_label: grp[y_colname].value_counts()
      for x_label, grp in df.groupby(x_colname)
  })
  sns.heatmap(df_2dhist, cmap=mpl_palette_name)
  plt.xlabel(x_colname)
  plt.ylabel(y_colname)
  return autoviz.MplChart.from_current_mpl_state()

chart3 = heatmap(foraging_df, *['time_of_day', 'island'], **{})

### Creature Pastimes by Island

In [ ]:
import numpy as np
from google.colab import autoviz

def heatmap(df, x_colname, y_colname, figscale=1, mpl_palette_name='viridis'):
  from matplotlib import pyplot as plt
  import seaborn as sns
  import pandas as pd
  plt.subplots(figsize=(8 * figscale, 8 * figscale))
  df_2dhist = pd.DataFrame({
      x_label: grp[y_colname].value_counts()
      for x_label, grp in df.groupby(x_colname)
  })
  sns.heatmap(df_2dhist, cmap=mpl_palette_name)
  plt.xlabel(x_colname)
  plt.ylabel(y_colname)
  return autoviz.MplChart.from_current_mpl_state()

chart = heatmap(migration_df, *['pastime', 'island'], **{})
chart

# 0. random number tools

In [ ]:
# importing a 'random' library that has lots of tools
import random

## generating random numbers or values

In [ ]:
# generate a random number
r = random.random()

# print out that number
r

In [ ]:
# random integer between a minimum and maximum value
random_health = random.randint(0, 100)

# print out that number
random_health

In [ ]:
# roll 0 or 1
rain = random.randint(0, 1)

# print out that number
rain

In [ ]:
# roll a dice
dice = random.randint(1, 6)

# print out that number
dice

In [ ]:
# pick a random choice from a list
choices = ["plant", "bug", "mushroom", "rock"]
resource = random.choice(choices)

# print out that choice
resource

In [ ]:
# generating a random weather: Rainy/not, Shady/not

day = [random.choice(["Rainy", "Normal"]), random.choice(["Shady", "Not Shady"])]
day

## generating multiple random numbers in a row or together

In [ ]:
# roll a 6-sided die 10 times
random_numbers = random.choices(range(1, 7), k=10)

random_numbers

In [ ]:
# another way to roll a 6-sided die 10 times
dice_rolls = []

# do ten times
for i in range(10):
  dice_rolls.append(random.randint(1,6))

random_numbers

In [ ]:
# generating 10 pairs of dice rolls, 6 sided and 20 sided

# create an empty list
pairs = []

# do ten times
for i in range(10):
  pairs.append(
      [random.randint(1, 6),
      random.randint(1, 20)]
  )

pairs

In [ ]:
# another way to write that
pairs = [[random.randint(1, 6), random.randint(1,20)] for _ in range(10)]

pairs

In [ ]:
# generating multiple days at random: Rainy/not, Shady/not
days = []
for i in range(10):
  day = [random.choice(["Rainy", "Normal"]), random.choice(["Shady", "Not Shady"])]
  days.append(day)

days

## weighting probabilities of picking certain values

In [ ]:
# weighted choice from a list
choices = ["plant", "bug", "mushroom", "rock"]

# weighted probabilities-- how likely each should be (as a fraction)
# rocks now most likely
weights = [0.1, 0.2, 0.3, 0.4]

# pick a single value from the list, given those weights
resource = random.choices(choices, weights, k=1)[0]

resource

In [ ]:
# weighted choices from a list
choices = ["plant", "bug", "mushroom", "rock"]

# weighted probabilities-- how likely each should be (as a fraction)
# rocks now most likely
weights = [0.1, 0.2, 0.3, 0.4]

# pick 10 using the weights
resources = random.choices(choices, weights, k=10)

resources

## changing probabilities based on variables

In [ ]:
# default or base weights
base_probabilities = [0.1, 0.2, 0.3, 0.4]

# multiply the original weights by some factor
# 1 keeps it the same, 2 doubles it, .5 cuts the probability in half
shade_multipliers = [.5, 2, 2, .5]

# modify to get new weights
modified_probabilities = []

# multiply values at each index in the list
for i, probability in enumerate(base_probabilities):
  modified_probabilities.append(probability * shade_multipliers[i])

modified_probabilities

## choosing a variable number of something

In [ ]:
# pick a variable amount from the weighted list

# how many to pick
faction_rep = 27

# weighted choices from a list
choices = ["plant", "bug", "mushroom", "rock"]

# weighted probabilities-- how likely each should be (as a fraction)
# rocks now most likely
weights = [0.1, 0.2, 0.3, 0.4]

# pick 10 using the weights
resources = random.choices(choices, weights, k=faction_rep)

resources

## creating a pattern or sequence

In [ ]:
# toggling a value between 0 and 1 each day

# 1 is "True" and 0 is "False"
# set default rain to False (not raining)
rain = 0

days = []
for i in range(10):
  days.append(rain)

  # flip the value of rain
  rain = 1 - rain

days

In [ ]:
# toggling a value between True and False each day

# 1 is "True" and 0 is "False"
# set default rain to False (not raining)
rain = False

days = []
for i in range(10):
  days.append(rain)

  # flip the value of rain
  rain = not rain

days

## enforce a rule or reset

In [ ]:
# enforcing a rule or reset (must rain once every 3 days)

all_days = []
last_two_days = ['null', 'null']

rain = random.randint(0,1)
for i in range(10):
  all_days.append(rain)

  # keep track of the last 2 days
  last_two_days[0] = last_two_days[1]
  last_two_days[1] = rain

  # set the next value of rain
  rain = random.randint(0,1)

  # but if it did not rain for last two days
  if not 1 in last_two_days:
    rain = 1

all_days